In [ ]:
# 1. Install the official Google GenAI SDK
%pip install -qU google-genai

# 2. Import libraries and load the key from Colab Secrets
from google.colab import userdata
from google.genai import client

# Retrieve the key (replace 'GEMINI_API_KEY' if you used a different name)
api_key = userdata.get('GEMINI_API_KEY')

# 3. Initialize the client and test with a simple prompt
try:
    cl = client.Client(api_key=api_key)
    response = cl.models.generate_content(
        model='gemini-2.5-flash-lite',
        contents='Say "Hello, your API key is working!" if you can read this.'
    )
    print("Success! Response from Gemini:")
    print(response.text)
except Exception as e:
    print(f"Error: Connection failed. Details: {e}")

Success! Response from Gemini:
Hello, your API key is working!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")
DRY_RUN = True          # set False after checking the printout

# keyword in current filename (lowercase) -> (doc_id, title, year, evidence_level)
DOCS = {
    "volume-3":              ("fao2021_recarb_v3", "Recarbonizing Global Soils, Volume 3: Cropland, grassland and integrated systems", 2021, "manual"),
    "vol 4":                 ("fao2021_recarb_v4", "Recarbonizing Global Soils, Volume 4: Case studies", 2021, "case_study"),
    "csrccl":                ("ipcc2019_srccl", "Climate Change and Land (IPCC Special Report)", 2019, "assessment"),
    "ipbes (2018)":          ("ipbes2018_ldr", "IPBES Assessment Report on Land Degradation and Restoration", 2018, "assessment"),
    "successful agroforestry": ("handa2019_agroforestry_models", "Successful Agroforestry Models for Different Agro-Ecological Regions in India", 2019, "manual"),
    "joshi":                 ("joshi2023_agj", "A global meta-analysis of cover crop response on soil carbon storage within a corn production system", 2023, "meta_analysis"),
    "soil pollution":        ("fao2018_soil_pollution", "Soil Pollution: A Hidden Reality", 2018, "assessment"),
    "livelihoodsecurity":    ("tewari_agroforestry_hot_arid", "Livelihood Improvements and Climate Change Adaptations Through Agroforestry in Hot Arid Environments", None, "review"),
    "isfr_book_eng-vol-1":   ("fsi2023_isfr_v1", "India State of Forest Report 2023, Volume 1", 2023, "assessment"),
    "isfr_book_eng-vol-2":   ("fsi2023_isfr_v2", "India State of Forest Report 2023, Volume 2", 2023, "assessment"),
    "frontiers in agronomy": ("keerthika2026_agroforestry_semiarid", "Multitier fruit-based agroforestry under deficit irrigation in semi-arid Rajasthan", 2026, "field_study"),
    "ca3129en":              ("fao2019_sowbfa", "The State of the World's Biodiversity for Food and Agriculture", 2019, "assessment"),
    "pollinators":           ("ipbes2016_pollinators", "IPBES Assessment Report on Pollinators, Pollination and Food Production", 2016, "assessment"),
    "gsocseq":               ("fao_gsocseq_intro", "Global Soil Organic Carbon Sequestration Potential Map (GSOCseq) - Introduction", None, "manual"),
}

registry, unmatched = {}, []

for pdf in sorted(ROOT.glob("p[01]/*.pdf")):
    hits = [v for k, v in DOCS.items() if k in pdf.name.lower()]
    if len(hits) != 1:
        unmatched.append(f"{pdf.parent.name}/{pdf.name}  ({len(hits)} matches)")
        continue

    doc_id, title, year, level = hits[0]
    target = pdf.with_name(doc_id + ".pdf")
    print(f"{pdf.parent.name}/{pdf.name[:50]:50} ->  {target.name}")

    if not DRY_RUN and target != pdf:
        if target.exists():
            print("   SKIPPED: target already exists")
            continue
        pdf.rename(target)

    registry[doc_id] = {
        "doc_id": doc_id,
        "title": title,
        "year": year,
        "evidence_level": level,
        "priority": pdf.parent.name,
        "path": str(target.relative_to(ROOT)),
        "needs_ocr_check": True,
        "pages_to_index": "all",
    }

print("\nUNMATCHED:", *unmatched, sep="\n  ") if unmatched else print("\nAll files matched.")

if not DRY_RUN:
    out = ROOT / "doc_registry.json"
    out.write_text(json.dumps(registry, indent=2, ensure_ascii=False))
    print(f"\nWrote {out} with {len(registry)} entries")

p0/Agronomy Journal - 2023 - Joshi - A global meta‐an ->  joshi2023_agj.pdf
p0/Climate_change_and_land_CSRCCL-.pdf                ->  ipcc2019_srccl.pdf
p0/IPBES (2018) Land Degradation and restoration asse ->  ipbes2018_ldr.pdf
p0/Recarbonizing global soils Volume-3.pdf            ->  fao2021_recarb_v3.pdf
p0/Recarbonizing global soils – Vol 4 Case studies.pd ->  fao2021_recarb_v4.pdf
p0/Successful Agroforestry Models for Different Agro- ->  handa2019_agroforestry_models.pdf
p1/1.Introduction_GSP_GSOCseq.pdf                     ->  fao_gsocseq_intro.pdf
p1/Assessment Report on Pollinators,.pdf              ->  ipbes2016_pollinators.pdf
p1/Frontiers in Agronomy.pdf                          ->  keerthika2026_agroforestry_semiarid.pdf
p1/LivelihoodSecurityEcosystemServicesAdvancesinAgrof ->  tewari_agroforestry_hot_arid.pdf
p1/Soil pollution a hidden reality.pdf                ->  fao2018_soil_pollution.pdf
p1/ca3129enThe State of the World's Biodiversity.pdf  ->  fao2019_sowbfa.pdf
p1/i

In [ ]:
for p in sorted(ROOT.glob("p[01]/*.pdf")):
    print(p.parent.name, "/", p.name)

p0 / Agronomy Journal - 2023 - Joshi - A global meta‐analysis of cover crop response on soil carbon storage within a corn.pdf
p0 / Climate_change_and_land_CSRCCL-.pdf
p0 / IPBES (2018) Land Degradation and restoration assesmet report.pdf
p0 / Recarbonizing global soils Volume-3.pdf
p0 / Recarbonizing global soils – Vol 4 Case studies.pdf
p0 / Successful Agroforestry Models for Different Agro-Ecological Regions in India.pdf
p1 / 1.Introduction_GSP_GSOCseq.pdf
p1 / Assessment Report on Pollinators,.pdf
p1 / Frontiers in Agronomy.pdf
p1 / LivelihoodSecurityEcosystemServicesAdvancesinAgroforestry10JCTewariPaper.pdf
p1 / Soil pollution a hidden reality.pdf
p1 / ca3129enThe State of the World's Biodiversity.pdf
p1 / isfr_book_eng-vol-1_2023.pdf
p1 / isfr_book_eng-vol-2_2023.pdf


In [ ]:
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")

DOCS = {
    "volume-3":                ("fao2021_recarb_v3", "Recarbonizing Global Soils, Volume 3: Cropland, grassland and integrated systems", 2021, "manual"),
    "vol 4":                   ("fao2021_recarb_v4", "Recarbonizing Global Soils, Volume 4: Case studies", 2021, "case_study"),
    "csrccl":                  ("ipcc2019_srccl", "Climate Change and Land (IPCC Special Report)", 2019, "assessment"),
    "ipbes (2018)":            ("ipbes2018_ldr", "IPBES Assessment Report on Land Degradation and Restoration", 2018, "assessment"),
    "successful agroforestry": ("handa2019_agroforestry_models", "Successful Agroforestry Models for Different Agro-Ecological Regions in India", 2019, "manual"),
    "joshi":                   ("joshi2023_agj", "A global meta-analysis of cover crop response on soil carbon storage within a corn production system", 2023, "meta_analysis"),
    "soil pollution":          ("fao2018_soil_pollution", "Soil Pollution: A Hidden Reality", 2018, "assessment"),
    "livelihoodsecurity":      ("tewari_agroforestry_hot_arid", "Livelihood Improvements and Climate Change Adaptations Through Agroforestry in Hot Arid Environments", None, "review"),
    "isfr_book_eng-vol-1":     ("fsi2023_isfr_v1", "India State of Forest Report 2023, Volume 1", 2023, "assessment"),
    "isfr_book_eng-vol-2":     ("fsi2023_isfr_v2", "India State of Forest Report 2023, Volume 2", 2023, "assessment"),
    "frontiers in agronomy":   ("keerthika2026_agroforestry_semiarid", "Multitier fruit-based agroforestry under deficit irrigation in semi-arid Rajasthan", 2026, "field_study"),
    "ca3129en":                ("fao2019_sowbfa", "The State of the World's Biodiversity for Food and Agriculture", 2019, "assessment"),
    "pollinators":             ("ipbes2016_pollinators", "IPBES Assessment Report on Pollinators, Pollination and Food Production", 2016, "assessment"),
    "gsocseq":                 ("fao_gsocseq_intro", "Global Soil Organic Carbon Sequestration Potential Map (GSOCseq) - Introduction", None, "manual"),
}

EXCLUDE_ESTIMATES = {"fsi2023_isfr_v1", "fsi2023_isfr_v2", "fao_gsocseq_intro"}

registry, unmatched = {}, []

for pdf in sorted(ROOT.glob("p[01]/*.pdf")):
    hits = [v for k, v in DOCS.items() if k in pdf.name.lower()]
    if len(hits) != 1:
        unmatched.append(f"{pdf.parent.name}/{pdf.name} ({len(hits)} matches)")
        continue

    doc_id, title, year, level = hits[0]
    target = pdf.with_name(doc_id + ".pdf")

    if target != pdf:
        if target.exists():
            print("SKIP, target exists:", target.name)
            continue
        pdf.rename(target)
        print(f"renamed -> {pdf.parent.name}/{target.name}")

    registry[doc_id] = {
        "doc_id": doc_id, "title": title, "year": year,
        "evidence_level": level, "priority": pdf.parent.name,
        "path": f"{pdf.parent.name}/{target.name}",
        "climate_zones": [], "practices": [], "metrics": [],
        "pages_to_index": "all", "needs_ocr_check": True,
        "exclude_from": ["variable_links", "estimates"] if doc_id in EXCLUDE_ESTIMATES else [],
    }

print("\nUNMATCHED:", unmatched or "none")

out = ROOT / "doc_registry.json"
out.write_text(json.dumps(registry, indent=2, ensure_ascii=False))
print(f"Wrote {out} ({len(registry)} entries)")

renamed -> p0/joshi2023_agj.pdf
renamed -> p0/ipcc2019_srccl.pdf
renamed -> p0/ipbes2018_ldr.pdf
renamed -> p0/fao2021_recarb_v3.pdf
renamed -> p0/fao2021_recarb_v4.pdf
renamed -> p0/handa2019_agroforestry_models.pdf
renamed -> p1/fao_gsocseq_intro.pdf
renamed -> p1/ipbes2016_pollinators.pdf
renamed -> p1/keerthika2026_agroforestry_semiarid.pdf
renamed -> p1/tewari_agroforestry_hot_arid.pdf
renamed -> p1/fao2018_soil_pollution.pdf
renamed -> p1/fao2019_sowbfa.pdf
renamed -> p1/fsi2023_isfr_v1.pdf
renamed -> p1/fsi2023_isfr_v2.pdf

UNMATCHED: none
Wrote /content/drive/MyDrive/Biodiversity_rag_data/doc_registry.json (14 entries)


In [ ]:
for p in sorted(ROOT.glob("p[01]/*.pdf")):
    print(p.parent.name, "/", p.name)

p0 / fao2021_recarb_v3.pdf
p0 / fao2021_recarb_v4.pdf
p0 / handa2019_agroforestry_models.pdf
p0 / ipbes2018_ldr.pdf
p0 / ipcc2019_srccl.pdf
p0 / joshi2023_agj.pdf
p1 / fao2018_soil_pollution.pdf
p1 / fao2019_sowbfa.pdf
p1 / fao_gsocseq_intro.pdf
p1 / fsi2023_isfr_v1.pdf
p1 / fsi2023_isfr_v2.pdf
p1 / ipbes2016_pollinators.pdf
p1 / keerthika2026_agroforestry_semiarid.pdf
p1 / tewari_agroforestry_hot_arid.pdf


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.6 MB/s eta 0:00:00


In [ ]:
import json
from pathlib import Path
from pypdf import PdfReader, PdfWriter

ROOT  = Path("/content/drive/MyDrive/Biodiversity_rag_data")
OUT   = ROOT / "p0_units"
TIERS = {"A", "B"}          # use {"A"} for a quick first pass

INDEX = {
 "joshi2023_agj": {"path": "p0/joshi2023_agj.pdf", "n_pages": 14, "offset": 0, "split": "unit",
  "units": [
   {"unit_id": "joshi2023_agj__full", "title": "Full paper", "s": 1, "e": 14, "practices": ["cover_crops"], "climate_zones": ["global"], "tier": "A"},
  ]},

 "fao2021_recarb_v3": {"path": "p0/fao2021_recarb_v3.pdf", "n_pages": 652, "offset": 18, "split": "unit",
  "units": [
   {"unit_id": "v3_p01_cover_cropping",      "title": "1. Cover cropping",                   "s": 20,  "e": 32,  "practices": ["cover_crops"], "tier": "A"},
   {"unit_id": "v3_p02_organic_mulch",       "title": "2. Organic mulch",                    "s": 32,  "e": 44,  "practices": ["mulching"], "tier": "A"},
   {"unit_id": "v3_p03_crop_rotations",      "title": "3. Crop rotations",                   "s": 44,  "e": 56,  "practices": ["crop_rotation"], "tier": "A"},
   {"unit_id": "v3_p04_intercropping_multi", "title": "4. Intercropping: multiple cropping", "s": 56,  "e": 66,  "practices": ["intercropping"], "tier": "A"},
   {"unit_id": "v3_p05_intercropping_strip", "title": "5. Intercropping: strip cropping",    "s": 66,  "e": 73,  "practices": ["intercropping"], "tier": "A"},
   {"unit_id": "v3_p28_irrigation",          "title": "28. Adequate irrigation practices",   "s": 363, "e": 375, "practices": ["irrigation"], "tier": "A"},
   {"unit_id": "v3_p38_agrisilvicultural",   "title": "38. Agroforestry 1: Agrisilvicultural","s": 491,"e": 504, "practices": ["agroforestry_agrisilvicultural"], "tier": "A"},
   {"unit_id": "v3_p39_silvopastoral",       "title": "39. Agroforestry 2: Silvopastoral",   "s": 504, "e": 517, "practices": ["agroforestry_silvopastoral"], "tier": "A"},
   {"unit_id": "v3_p40_agrosilvopastoral",   "title": "40. Agroforestry 3: Agrosilvopastoral","s": 517,"e": 529, "practices": ["agroforestry_agrosilvopastoral"], "tier": "A"},
   {"unit_id": "v3_p06_no_till",             "title": "6. No-till",                          "s": 73,  "e": 90,  "practices": ["no_till"], "tier": "B"},
   {"unit_id": "v3_p07_reduced_tillage",     "title": "7. Conservation, reduced tillage",    "s": 90,  "e": 103, "practices": ["reduced_tillage"], "tier": "B"},
   {"unit_id": "v3_p10_manure",              "title": "10. Manure additions",                "s": 122, "e": 135, "practices": ["manure"], "tier": "B"},
   {"unit_id": "v3_p12_compost",             "title": "12. Compost application",             "s": 146, "e": 156, "practices": ["compost"], "tier": "B"},
   {"unit_id": "v3_p16_isfm",                "title": "16. Integrated soil fertility mgmt",  "s": 207, "e": 224, "practices": ["integrated_nutrient_mgmt"], "tier": "B"},
   {"unit_id": "v3_p22_gypsum_sodic",        "title": "22. Gypsum on sodic soils",           "s": 305, "e": 313, "practices": ["gypsum_amendment"], "tier": "B"},
   {"unit_id": "v3_p23_terraces",            "title": "23. Terraces",                        "s": 313, "e": 324, "practices": ["terracing"], "tier": "B"},
   {"unit_id": "v3_p24_check_dams",          "title": "24. Check dams",                      "s": 324, "e": 331, "practices": ["check_dams"], "tier": "B"},
   {"unit_id": "v3_p25_shelterbelts",        "title": "25. Shelterbelts",                    "s": 331, "e": 339, "practices": ["shelterbelts"], "tier": "B"},
   {"unit_id": "v3_p26_hedges_buffers",      "title": "26. Hedges and buffer strips",        "s": 339, "e": 354, "practices": ["hedges_buffer_strips"], "tier": "B"},
   {"unit_id": "v3_p32_grassland_restore",   "title": "32. Restoration of degraded grassland","s": 415,"e": 423, "practices": ["grassland_restoration"], "tier": "B"},
   {"unit_id": "v3_p35_grazing_mgmt",        "title": "35. Grazing exclusion, rotational grazing","s": 450,"e": 462,"practices": ["rotational_grazing"], "tier": "B"},
   {"unit_id": "v3_p37_crop_livestock",      "title": "37. Integrated crop-livestock",       "s": 473, "e": 491, "practices": ["crop_livestock_integration"], "tier": "B"},
   {"unit_id": "v3_p42_conservation_ag",     "title": "42. Conservation agriculture",        "s": 541, "e": 565, "practices": ["conservation_agriculture"], "tier": "B"},
  ]},

 "fao2021_recarb_v4": {"path": "p0/fao2021_recarb_v4.pdf", "n_pages": 550, "offset": 24, "split": "unit",
  "units": [
   {"unit_id": "v4_c01_notill_olive_lebanon",   "title": "1. No-tillage olive orchards, Lebanon",       "s": 27,  "e": 36,  "practices": ["no_till"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c05_ca_south_africa",        "title": "5. Conservation agriculture, South Africa",   "s": 72,  "e": 92,  "practices": ["conservation_agriculture"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c06_intercropping_africa",   "title": "6. Legume-cereal intercropping, Africa",      "s": 92,  "e": 101, "practices": ["intercropping"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c17_dehesa_iberia",          "title": "17. Agrosilvopastoral savanna, Iberia",       "s": 210, "e": 220, "practices": ["agroforestry_agrosilvopastoral"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c20_mulch_granada",          "title": "20. Mulching subtropical orchards, Granada",  "s": 237, "e": 247, "practices": ["mulching"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c21_covercrop_almond_spain", "title": "21. Reduced tillage + cover crops, rainfed almond","s": 247,"e": 256,"practices": ["cover_crops", "reduced_tillage"], "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "v4_c02_restoration_madagascar", "title": "2. Soil ecological restoration, Madagascar",  "s": 36,  "e": 45,  "practices": ["conservation_agriculture"], "tier": "B"},
   {"unit_id": "v4_c09_grazing_australia",      "title": "9. Rangeland grazing management, Australia",  "s": 130, "e": 139, "practices": ["rotational_grazing"], "climate_zones": ["arid", "semi-arid"], "tier": "B"},
   {"unit_id": "v4_c11_mulch_thailand",         "title": "11. Rice straw mulching, no-till, Thailand",  "s": 149, "e": 160, "practices": ["mulching", "no_till"], "tier": "B"},
   {"unit_id": "v4_c16_olive_italy",            "title": "16. Mediterranean olive orchard, Italy",      "s": 200, "e": 210, "practices": ["cover_crops"], "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "v4_c25_irrigated_harran",       "title": "25. Irrigated wheat-maize-cotton, Turkey",    "s": 291, "e": 301, "practices": ["irrigation"], "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "v4_c36_agroforestry_colombia",  "title": "36. Agroforestry, silvopastoral, Colombia",   "s": 427, "e": 439, "practices": ["agroforestry_silvopastoral"], "tier": "B"},
   {"unit_id": "v4_c46_deficit_irrigation_ks",  "title": "46. Deficit irrigation, Kansas",              "s": 532, "e": 540, "practices": ["irrigation"], "climate_zones": ["semi-arid"], "tier": "B"},
  ]},

 "handa2019_agroforestry_models": {"path": "p0/handa2019_agroforestry_models.pdf", "n_pages": 225, "offset": 18, "split": "unit",
  "units": [
   {"unit_id": "handa_2_1_bakain_agrisilvi",      "title": "AER2 2.1 Bakain agri-silvicultural",   "s": 25,  "e": 28,  "climate_zones": ["arid"], "tier": "A"},
   {"unit_id": "handa_3_1_melia_dubia",           "title": "AER3 3.1 Melia dubia (400-500 mm)",    "s": 28,  "e": 33,  "climate_zones": ["arid"], "rainfall_mm": [400, 500], "tier": "A"},
   {"unit_id": "handa_4_1_ailanthus_rainfed",     "title": "AER4 4.1 Ailanthus, rainfed",          "s": 33,  "e": 44,  "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "handa_4_2_shisham",               "title": "AER4 4.2 Shisham agri-silvi",          "s": 44,  "e": 49,  "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "handa_4_3_aonla_agrihorti",       "title": "AER4 4.3 Aonla agri-horticultural",    "s": 49,  "e": 54,  "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "handa_5_1_melia_azedarach",       "title": "AER5 5.1 Melia azedarach agri-silvi",  "s": 54,  "e": 58,  "climate_zones": ["semi-arid"], "tier": "A"},
   {"unit_id": "handa_1_1_mulberry_silvopastoral","title": "AER1 1.1 Mulberry silvi-pastoral",     "s": 19,  "e": 25,  "climate_zones": ["arid"], "tier": "B"},
   {"unit_id": "handa_6_1_three_tier_paddy",      "title": "AER6 6.1 Three-tier, paddy",           "s": 58,  "e": 62,  "climate_zones": ["semi-arid"], "rainfall_mm": [600, 1000], "tier": "B"},
   {"unit_id": "handa_6_2_teak_agrisilvi",        "title": "AER6 6.2 Teak agri-silvicultural",     "s": 62,  "e": 65,  "climate_zones": ["semi-arid"], "rainfall_mm": [600, 1000], "tier": "B"},
   {"unit_id": "handa_6_3_sapota_teak",           "title": "AER6 6.3 Sapota-teak horti-silvi",     "s": 65,  "e": 70,  "climate_zones": ["semi-arid"], "rainfall_mm": [600, 1000], "tier": "B"},
   {"unit_id": "handa_6_4_sapota_eucalyptus",     "title": "AER6 6.4 Sapota-eucalyptus, degraded", "s": 70,  "e": 74,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_6_5_tamarind_silvihorti",   "title": "AER6 6.5 Tamarind silvi-horti",        "s": 74,  "e": 77,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_6_6_tamarind_degraded",     "title": "AER6 6.6 Tamarind, degraded lands",    "s": 77,  "e": 80,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_7_1_terminalia",            "title": "AER7 7.1 Terminalia agri-silvi",       "s": 80,  "e": 82,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_7_2_tamarind",              "title": "AER7 7.2 Tamarind agri-silvi",         "s": 82,  "e": 84,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_7_3_mango_agrihorti",       "title": "AER7 7.3 Mango agri-horticultural",    "s": 84,  "e": 87,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_8_1_sandalwood",            "title": "AER8 8.1 Sandalwood block plantation", "s": 87,  "e": 89,  "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_8_2_melia_clonal",          "title": "AER8 8.2 Melia dubia clonal",          "s": 89,  "e": 102, "climate_zones": ["semi-arid"], "tier": "B"},
   {"unit_id": "handa_8_3_coconut_hortipastoral", "title": "AER8 8.3 Coconut horti-pastoral",      "s": 102, "e": 109, "climate_zones": ["semi-arid"], "tier": "B"},
  ]},

 "ipcc2019_srccl": {"path": "p0/ipcc2019_srccl.pdf", "n_pages": 874, "offset": 10,
  "split": "window", "window": 12, "overlap": 1,
  "units": [
   {"unit_id": "srccl_ch3_desertification",  "title": "Ch3 Desertification",  "s": 259, "e": 356, "tier": "A"},
   {"unit_id": "srccl_ch4_land_degradation", "title": "Ch4 Land degradation", "s": 355, "e": 448, "tier": "A"},
   {"unit_id": "srccl_ch6_interlinkages",    "title": "Ch6 Interlinkages",    "s": 561, "e": 684, "tier": "A"},
  ]},

 "ipbes2018_ldr": {"path": "p0/ipbes2018_ldr.pdf", "n_pages": 748, "offset": 58,
  "split": "window", "window": 12, "overlap": 1,
  "units": [
   {"unit_id": "ipbes_ch4_status_trends", "title": "Ch4 Status, trends, biodiversity", "s": 279, "e": 400, "tier": "A"},
   {"unit_id": "ipbes_ch6_responses",     "title": "Ch6 Responses",                    "s": 493, "e": 590, "tier": "A"},
   {"unit_id": "ipbes_ch3_drivers",       "title": "Ch3 Drivers",                      "s": 195, "e": 280, "tier": "C"},
  ]},
}

def windows(s, e, size, ov):
    while s <= e:
        end = min(s + size - 1, e)
        yield s, end
        if end == e:
            break
        s = end - ov + 1

manifest, made, skipped = [], 0, 0

for doc_id, doc in INDEX.items():
    src = ROOT / doc["path"]
    if not src.exists():
        print("MISSING:", src); continue
    reader = PdfReader(str(src))
    n = len(reader.pages)
    if n != doc["n_pages"]:
        print(f"WARNING {doc_id}: expected {doc['n_pages']} pages, file has {n}")
    dest_dir = OUT / doc_id
    dest_dir.mkdir(parents=True, exist_ok=True)

    for u in doc["units"]:
        if u.get("tier", "A") not in TIERS:
            continue
        pieces = ([(u["s"], u["e"])] if doc["split"] != "window"
                  else list(windows(u["s"], u["e"], doc["window"], doc["overlap"])))
        for k, (s, e) in enumerate(pieces):
            s, e = max(1, s), min(e, n)
            uid = u["unit_id"] if len(pieces) == 1 else f"{u['unit_id']}__w{k:02d}"
            dest = dest_dir / f"{uid}.pdf"
            if dest.exists():
                skipped += 1
            else:
                w = PdfWriter()
                for i in range(s - 1, e):
                    w.add_page(reader.pages[i])
                with open(dest, "wb") as f:
                    w.write(f)
                made += 1
            manifest.append({
                "unit_id": uid, "doc_id": doc_id, "parent_unit": u["unit_id"],
                "title": u["title"], "pdf_start": s, "pdf_end": e, "n_pages": e - s + 1,
                "printed_start": s - doc["offset"], "path": str(dest.relative_to(ROOT)),
                "practices": u.get("practices", []), "climate_zones": u.get("climate_zones", []),
                "rainfall_mm": u.get("rainfall_mm"), "tier": u.get("tier", "A"),
            })

(ROOT / "p0_units_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"created {made}, existed {skipped}, total units {len(manifest)}")

created 19, existed 87, total units 106


In [ ]:
for uid in ["v3_p01_cover_cropping", "handa_3_1_melia_dubia", "v4_c21_covercrop_almond_spain"]:
    rec = next(m for m in manifest if m["unit_id"] == uid)
    txt = PdfReader(str(ROOT / rec["path"])).pages[0].extract_text() or ""
    print(uid, "->", txt[:110].replace("\n", " "))

v3_p01_cover_cropping -> RECARBONIZING GLOBAL SOILS 2  CROPLAND  SOIL ORGANIC COVER  1. Cover cropping Rosa Francaviglia1, José Luis Vi
handa_3_1_melia_dubia -> 10 Habit and Habitat It is a fairly large, handsome, deciduous tree, attaining a girth of  1.2-1.5  m and a he
v4_c21_covercrop_almond_spain ->   VOLUME 4: CROPLAND, GRASSLAND, INTEGRATED SYSTEMS AND FARMING APPROACHES   CASE STUDIES  223  21. Reduced ti


In [ ]:
import shutil
shutil.rmtree(ROOT / "p0_units" / "handa2019_agroforestry_models", ignore_errors=True)

In [ ]:
for uid in ["handa_3_1_melia_dubia", "handa_4_1_ailanthus_rainfed", "handa_5_1_melia_azedarach"]:
    rec = next(m for m in manifest if m["unit_id"] == uid)
    print(uid, "->", (PdfReader(str(ROOT / rec["path"])).pages[0].extract_text() or "")[:110].replace("\n", " "))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Biodiversity_rag_data/p0_units/handa2019_agroforestry_models/handa_3_1_melia_dubia.pdf'

In [ ]:
import json
from pathlib import Path
from pypdf import PdfReader

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")
mani = json.loads((ROOT / "p0_units_manifest.json").read_text())
by_id = {m["unit_id"]: m for m in mani}

for uid in ["handa_1_1_mulberry_silvopastoral", "handa_3_1_melia_dubia",
            "handa_4_1_ailanthus_rainfed", "handa_5_1_melia_azedarach"]:
    rec = by_id.get(uid)
    if not rec:
        print(uid, "-> not in manifest"); continue
    f = ROOT / rec["path"]
    if not f.exists():
        print(uid, "-> file missing, re-run the splitter"); continue
    txt = (PdfReader(str(f)).pages[0].extract_text() or "").replace("\n", " ")
    print(f"{uid:38} p{rec['pdf_start']:>4} -> {txt[:100]}")

handa_1_1_mulberry_silvopastoral -> file missing, re-run the splitter
handa_3_1_melia_dubia -> file missing, re-run the splitter
handa_4_1_ailanthus_rainfed -> file missing, re-run the splitter
handa_5_1_melia_azedarach -> file missing, re-run the splitter


In [ ]:
INDEX["handa2019_agroforestry_models"]["offset"] = 17
for u in INDEX["handa2019_agroforestry_models"]["units"]:
    u["s"] -= 1
print([(u["unit_id"], u["s"], u["e"]) for u in INDEX["handa2019_agroforestry_models"]["units"]][:4])

[('handa_2_1_bakain_agrisilvi', 24, 28), ('handa_3_1_melia_dubia', 27, 33), ('handa_4_1_ailanthus_rainfed', 32, 44), ('handa_4_2_shisham', 43, 49)]


In [ ]:
import json
from pathlib import Path
from pypdf import PdfReader

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")
mani = json.loads((ROOT / "p0_units_manifest.json").read_text())
by_id = {m["unit_id"]: m for m in mani}

for uid in ["handa_2_1_bakain_agrisilvi", "handa_3_1_melia_dubia",
            "handa_4_1_ailanthus_rainfed", "handa_5_1_melia_azedarach"]:
    rec = by_id[uid]
    txt = (PdfReader(str(ROOT / rec["path"])).pages[0].extract_text() or "").replace("\n", " ")
    print(f"{uid:34} p{rec['pdf_start']:>4} -> {txt[:100]}")

handa_2_1_bakain_agrisilvi         p  25 -> 7 Habit and Habitat Melia azedarach is a small to medium, deciduous tree from 6 to  35 m in height. 
handa_3_1_melia_dubia              p  28 -> 10 Habit and Habitat It is a fairly large, handsome, deciduous tree, attaining a girth of  1.2-1.5  
handa_4_1_ailanthus_rainfed        p  33 -> 15 Habit and Habitat Ailanthus excelsa grows well in semi-arid and semi-moist regions  and has been 
handa_5_1_melia_azedarach          p  54 -> 36 Habit and Habitat Melia azedarach   is a fast growing deciduous to semi-evergreen  tree. The adul


In [ ]:
import json, shutil
from pathlib import Path
from pypdf import PdfReader, PdfWriter

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")
DOC_ID = "handa2019_agroforestry_models"
SRC = ROOT / "p0" / f"{DOC_ID}.pdf"
DEST_DIR = ROOT / "p0_units" / DOC_ID
OFFSET = 17

UNITS = [
    ("handa_1_1_mulberry_silvopastoral", "AER1 1.1 Mulberry silvi-pastoral",      18,  25, ["arid"], None, "B"),
    ("handa_2_1_bakain_agrisilvi",       "AER2 2.1 Bakain agri-silvicultural",    24,  28, ["arid"], None, "A"),
    ("handa_3_1_melia_dubia",            "AER3 3.1 Melia dubia (400-500 mm)",     27,  33, ["arid"], [400, 500], "A"),
    ("handa_4_1_ailanthus_rainfed",      "AER4 4.1 Ailanthus, rainfed",           32,  44, ["semi-arid"], None, "A"),
    ("handa_4_2_shisham",                "AER4 4.2 Shisham agri-silvi",           43,  49, ["semi-arid"], None, "A"),
    ("handa_4_3_aonla_agrihorti",        "AER4 4.3 Aonla agri-horticultural",     48,  54, ["semi-arid"], None, "A"),
    ("handa_5_1_melia_azedarach",        "AER5 5.1 Melia azedarach agri-silvi",   53,  58, ["semi-arid"], None, "A"),
    ("handa_6_1_three_tier_paddy",       "AER6 6.1 Three-tier, paddy",            57,  62, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_2_teak_agrisilvi",         "AER6 6.2 Teak agri-silvicultural",      61,  65, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_3_sapota_teak",            "AER6 6.3 Sapota-teak horti-silvi",      64,  70, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_4_sapota_eucalyptus",      "AER6 6.4 Sapota-eucalyptus, degraded",  69,  74, ["semi-arid"], None, "B"),
    ("handa_6_5_tamarind_silvihorti",    "AER6 6.5 Tamarind silvi-horti",         73,  77, ["semi-arid"], None, "B"),
    ("handa_6_6_tamarind_degraded",      "AER6 6.6 Tamarind, degraded lands",     76,  80, ["semi-arid"], None, "B"),
    ("handa_7_1_terminalia",             "AER7 7.1 Terminalia agri-silvi",        79,  82, ["semi-arid"], None, "B"),
    ("handa_7_2_tamarind",               "AER7 7.2 Tamarind agri-silvi",          81,  84, ["semi-arid"], None, "B"),
    ("handa_7_3_mango_agrihorti",        "AER7 7.3 Mango agri-horticultural",     83,  87, ["semi-arid"], None, "B"),
    ("handa_8_1_sandalwood",             "AER8 8.1 Sandalwood block plantation",  86,  89, ["semi-arid"], None, "B"),
    ("handa_8_2_melia_clonal",           "AER8 8.2 Melia dubia clonal",           88, 102, ["semi-arid"], None, "B"),
    ("handa_8_3_coconut_hortipastoral",  "AER8 8.3 Coconut horti-pastoral",      101, 109, ["semi-arid"], None, "B"),
]

shutil.rmtree(DEST_DIR, ignore_errors=True)
DEST_DIR.mkdir(parents=True, exist_ok=True)

reader = PdfReader(str(SRC))
n = len(reader.pages)
new_entries = []

for uid, title, s, e, zones, rain, tier in UNITS:
    s, e = max(1, s), min(e, n)
    dest = DEST_DIR / f"{uid}.pdf"
    w = PdfWriter()
    for i in range(s - 1, e):
        w.add_page(reader.pages[i])
    with open(dest, "wb") as f:
        w.write(f)
    new_entries.append({
        "unit_id": uid, "doc_id": DOC_ID, "parent_unit": uid, "title": title,
        "pdf_start": s, "pdf_end": e, "n_pages": e - s + 1,
        "printed_start": s - OFFSET, "path": str(dest.relative_to(ROOT)),
        "practices": [], "climate_zones": zones, "rainfall_mm": rain, "tier": tier,
    })

mf = ROOT / "p0_units_manifest.json"
mani = [m for m in json.loads(mf.read_text()) if m["doc_id"] != DOC_ID] + new_entries
mf.write_text(json.dumps(mani, indent=2))
print(f"rewrote {len(new_entries)} handa units, manifest now {len(mani)} units")

for uid in ["handa_2_1_bakain_agrisilvi", "handa_3_1_melia_dubia",
            "handa_4_1_ailanthus_rainfed", "handa_5_1_melia_azedarach"]:
    rec = next(m for m in mani if m["unit_id"] == uid)
    txt = (PdfReader(str(ROOT / rec["path"])).pages[0].extract_text() or "").replace("\n", " ")
    print(f"{uid:34} p{rec['pdf_start']:>4} -> {txt[:100]}")

rewrote 19 handa units, manifest now 106 units
handa_2_1_bakain_agrisilvi         p  24 -> 6 Silvi-pasture system on an average cycle of 10 years could  generate 120 person days\ha\yr employm
handa_3_1_melia_dubia              p  27 -> 9 2.1. Agroforestry Model: Bakain (Melia azedarach) based Agri-silvicultural System Utilization: Tim
handa_4_1_ailanthus_rainfed        p  32 -> 14 Intercrops yield Crop Yield (x100 kg/ha) First  year Second  year Third  year Fourth  year Fifth 
handa_5_1_melia_azedarach          p  53 -> 35 Benefits Accrued to Farmers/Public The Aonla based agri-horticulture system has been proven to be


In [ ]:
import json, shutil
from pathlib import Path
from pypdf import PdfReader, PdfWriter

ROOT = Path("/content/drive/MyDrive/Biodiversity_rag_data")
DOC_ID = "handa2019_agroforestry_models"
SRC = ROOT / "p0" / f"{DOC_ID}.pdf"
DEST_DIR = ROOT / "p0_units" / DOC_ID
OFFSET = 18

UNITS = [
    ("handa_1_1_mulberry_silvopastoral", "AER1 1.1 Mulberry silvi-pastoral",     19,  25, ["arid"], None, "B"),
    ("handa_2_1_bakain_agrisilvi",       "AER2 2.1 Bakain agri-silvicultural",   25,  28, ["arid"], None, "A"),
    ("handa_3_1_melia_dubia",            "AER3 3.1 Melia dubia (400-500 mm)",    28,  33, ["arid"], [400, 500], "A"),
    ("handa_4_1_ailanthus_rainfed",      "AER4 4.1 Ailanthus, rainfed",          33,  44, ["semi-arid"], None, "A"),
    ("handa_4_2_shisham",                "AER4 4.2 Shisham agri-silvi",          44,  49, ["semi-arid"], None, "A"),
    ("handa_4_3_aonla_agrihorti",        "AER4 4.3 Aonla agri-horticultural",    49,  54, ["semi-arid"], None, "A"),
    ("handa_5_1_melia_azedarach",        "AER5 5.1 Melia azedarach agri-silvi",  54,  58, ["semi-arid"], None, "A"),
    ("handa_6_1_three_tier_paddy",       "AER6 6.1 Three-tier, paddy",           58,  62, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_2_teak_agrisilvi",         "AER6 6.2 Teak agri-silvicultural",     62,  65, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_3_sapota_teak",            "AER6 6.3 Sapota-teak horti-silvi",     65,  70, ["semi-arid"], [600, 1000], "B"),
    ("handa_6_4_sapota_eucalyptus",      "AER6 6.4 Sapota-eucalyptus, degraded", 70,  74, ["semi-arid"], None, "B"),
    ("handa_6_5_tamarind_silvihorti",    "AER6 6.5 Tamarind silvi-horti",        74,  77, ["semi-arid"], None, "B"),
    ("handa_6_6_tamarind_degraded",      "AER6 6.6 Tamarind, degraded lands",    77,  80, ["semi-arid"], None, "B"),
    ("handa_7_1_terminalia",             "AER7 7.1 Terminalia agri-silvi",       80,  82, ["semi-arid"], None, "B"),
    ("handa_7_2_tamarind",               "AER7 7.2 Tamarind agri-silvi",         82,  84, ["semi-arid"], None, "B"),
    ("handa_7_3_mango_agrihorti",        "AER7 7.3 Mango agri-horticultural",    84,  87, ["semi-arid"], None, "B"),
    ("handa_8_1_sandalwood",             "AER8 8.1 Sandalwood block plantation", 87,  89, ["semi-arid"], None, "B"),
    ("handa_8_2_melia_clonal",           "AER8 8.2 Melia dubia clonal",          89, 102, ["semi-arid"], None, "B"),
    ("handa_8_3_coconut_hortipastoral",  "AER8 8.3 Coconut horti-pastoral",     102, 109, ["semi-arid"], None, "B"),
]

shutil.rmtree(DEST_DIR, ignore_errors=True); DEST_DIR.mkdir(parents=True, exist_ok=True)
reader = PdfReader(str(SRC)); n = len(reader.pages); new = []

for uid, title, s, e, zones, rain, tier in UNITS:
    s, e = max(1, s), min(e, n)
    dest = DEST_DIR / f"{uid}.pdf"
    w = PdfWriter()
    for i in range(s - 1, e):
        w.add_page(reader.pages[i])
    with open(dest, "wb") as f:
        w.write(f)
    new.append({"unit_id": uid, "doc_id": DOC_ID, "parent_unit": uid, "title": title,
                "pdf_start": s, "pdf_end": e, "n_pages": e - s + 1,
                "printed_start": s - OFFSET, "path": str(dest.relative_to(ROOT)),
                "practices": [], "climate_zones": zones, "rainfall_mm": rain, "tier": tier})

mf = ROOT / "p0_units_manifest.json"
mani = [m for m in json.loads(mf.read_text()) if m["doc_id"] != DOC_ID] + new
mf.write_text(json.dumps(mani, indent=2))
print("handa units rewritten:", len(new), "| manifest:", len(mani))

# proper check: search the WHOLE first page for a model heading
import re
for rec in sorted([m for m in mani if m["doc_id"] == DOC_ID], key=lambda r: r["pdf_start"]):
    txt = (PdfReader(str(ROOT / rec["path"])).pages[0].extract_text() or "").replace("\n", " ")
    hit = re.search(r"\d+\.\d+\.?\s*Agroforestry Model[^.]{0,70}", txt)
    print(f"{rec['unit_id']:34} p{rec['pdf_start']:>4} -> {hit.group(0)[:70] if hit else 'NO HEADING ON PAGE 1'}")

handa units rewritten: 19 | manifest: 106
handa_1_1_mulberry_silvopastoral   p  19 -> 1.1 Agroforestry Model: Mulberry based Silvi- pastural System Area of 
handa_2_1_bakain_agrisilvi         p  25 -> 2.1. Agroforestry Model: Bakain (Melia azedarach)  based Agri-silvicul
handa_3_1_melia_dubia              p  28 -> 3.1 Agroforestry Model: Melia dubia based Agri-sylvicultural System Ar
handa_4_1_ailanthus_rainfed        p  33 -> 4.1 Agroforestry Model:  Ailanthus based Agri- silvicultural System un
handa_4_2_shisham                  p  44 -> 4.2. Agroforestry Model: Shisham based Agri- silvicultural and Silvi-p
handa_4_3_aonla_agrihorti          p  49 -> 4.3. Agroforestry Model: Aonla based Agri-horticultural System Area of
handa_5_1_melia_azedarach          p  54 -> 5.1. Agroforestry Model: Melia (Melia azedarach)  based Agroforestry S
handa_6_1_three_tier_paddy         p  58 -> 6.1. Agroforestry Model: Three-tier Agroforestry  System for Paddy Gro
handa_6_2_teak_agrisilvi           p  

### PDF Extrcation

In [ ]:
!pip -q install google-genai pydantic pypdf

import os, json, time, random, hashlib, re
from pathlib import Path
from typing import Optional, List, Literal
from google import genai
from google.genai import types
from pydantic import BaseModel
from google.colab import userdata, drive
from pypdf import PdfReader

drive.mount('/content/drive', force_remount=False)

ROOT      = Path("/content/drive/MyDrive/Biodiversity_rag_data")
UNITS_DIR = ROOT / "p0_units"
OUT_DIR   = ROOT / "extracted"
FAIL_DIR  = OUT_DIR / "_failures"
OUT_DIR.mkdir(exist_ok=True); FAIL_DIR.mkdir(exist_ok=True)

MODEL          = "gemini-3.1-flash-lite"
FALLBACK_MODEL = "gemini-2.5-flash-lite"
PROMPT_VERSION = "v1.0"
SLEEP_BETWEEN  = 5.0        # seconds between units (free tier is ~15 RPM)

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
manifest = json.loads((ROOT / "p0_units_manifest.json").read_text())
print(len(manifest), "units in manifest")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
106 units in manifest


In [ ]:
METRICS = Literal["SOC","soil_organic_matter","pH","soil_moisture","water_holding_capacity",
                  "bulk_density","erosion","nutrient_availability","nitrogen","microbial_biomass",
                  "soil_biota","species_richness","habitat_diversity","fragmentation","pollinators",
                  "tree_cover","biomass","crop_yield","temperature","rainfall","salinity",
                  "pollution_load","carbon_sequestration_rate"]

PRACTICES = Literal["cover_crops","mulching","crop_rotation","intercropping","no_till",
                    "reduced_tillage","manure","compost","integrated_nutrient_mgmt","irrigation",
                    "deficit_irrigation","terracing","check_dams","shelterbelts",
                    "hedges_buffer_strips","grassland_restoration","rotational_grazing",
                    "crop_livestock_integration","conservation_agriculture",
                    "agroforestry_agrisilvicultural","agroforestry_silvopastoral",
                    "agroforestry_agrosilvopastoral","agroforestry_horti_pastoral",
                    "gypsum_amendment","other"]

ZONES     = Literal["arid","semi-arid","dry_sub_humid","humid","temperate","tropical","global","unstated"]
LAND_USE  = Literal["cropland","grassland","forest","orchard","rangeland","degraded_land",
                    "integrated_system","unstated"]
UNITS_ENUM= Literal["percent","percent_change","Mg_ha","Mg_ha_yr","g_kg","kg_ha","t_ha",
                    "mm","ratio","index","count","none"]

class Passage(BaseModel):
    passage_id: str
    heading_path: List[str]
    text: str
    page_start: int
    page_end: int
    topic: str
    practices: List[PRACTICES]
    metrics: List[METRICS]
    climate_zones: List[ZONES]
    land_use: List[LAND_USE]
    content_role: Literal["description","mechanism","evidence","constraint","risk","context","table"]

class Claim(BaseModel):
    claim_id: str
    passage_id: str
    subject: str
    metric: METRICS
    direction: Literal["increase","decrease","no_change","mixed"]
    value: Optional[float] = None
    value_low: Optional[float] = None
    value_high: Optional[float] = None
    unit: Optional[UNITS_ENUM] = None
    basis: Optional[str] = None
    conditions: Optional[str] = None
    time_horizon_years: Optional[float] = None
    evidence_span: str
    confidence_in_extraction: Literal["explicit","derived"]

class Link(BaseModel):
    from_node: str
    to_node: str
    effect: Literal["increase","decrease","no_change","mixed"]
    condition: Optional[str] = None
    mechanism: Optional[str] = None
    passage_id: str
    strength_stated: Literal["strong","moderate","weak","unstated"]

class PracticeProfile(BaseModel):
    system_name: str
    system_type: Optional[str] = None
    tree_species: List[str]
    crop_species: List[str]
    region_of_adoption: List[str]
    rainfall_mm_low: Optional[int] = None
    rainfall_mm_high: Optional[int] = None
    soil_requirements: Optional[str] = None
    establishment_notes: Optional[str] = None
    stated_benefits: List[str]
    stated_constraints: List[str]

class UnitExtraction(BaseModel):
    extraction_status: Literal["ok","partial","no_content"]
    section_title: str
    content_type: Literal["practice","case_study","assessment","model","methods","other"]
    summary: str
    passages: List[Passage]
    claims: List[Claim]
    links: List[Link]
    practice_profile: Optional[PracticeProfile] = None

In [ ]:
SYSTEM_PROMPT = """You are a scientific data extraction system for an environmental knowledge base. You extract what a document states. You never infer, estimate, generalise, or use knowledge from outside the provided pages.

TASK
Read the attached PDF pages and return a single JSON object matching the provided schema.

RULE 1 — PASSAGES ARE VERBATIM
`text` must be copied character-for-character from the document. Never paraphrase, summarise, merge distant sentences, or fix wording inside `text`. Your judgement is used only to choose passage boundaries and to label them. A passage is one coherent block: typically 3-12 sentences, or one complete table rendered as readable rows.
Set `page_start`/`page_end` to the PRINTED page numbers shown on the page. If no printed number is visible, use the position in this PDF slice starting at 1.

RULE 2 — SKIP NON-CONTENT
Do not create passages from: reference lists, author or contributor lists, acknowledgements, tables of contents, figure captions with no substantive text, page headers and footers, or pages that are only photographs. If the pages contain no extractable scientific content, return extraction_status "no_content" with empty lists.

RULE 3 — CLAIMS NEED RECEIPTS
Create a claim only where the document states an effect on a metric. `evidence_span` must be the exact sentence or clause from the document containing that statement, copied verbatim. If a number appears, it must appear inside `evidence_span`.
- Use `value` for a point estimate, `value_low`/`value_high` for a stated range.
- `confidence_in_extraction`: "explicit" when the text states the value directly; "derived" when you computed or read it from a table. Never mark an inferred number "explicit".
- Record `conditions` exactly as stated (climate, soil type, duration, crop system). If conditions are not stated, use null. Do not invent them.
- A directional statement with no number is still a valid claim: set direction and leave value fields null.

RULE 4 — LINKS CARRY CONDITIONS
Create a link for each cause-effect relationship the document asserts: practice to metric, or metric to metric. The `condition` field is critical: if the document says an effect holds only under certain climates, soils, or durations, record that condition verbatim in short form (e.g. "arid|semi-arid", "sandy soils", "first 3 years"). If the effect is stated unconditionally, use null. Do not create links the document does not assert.

RULE 5 — PRACTICE PROFILE
Fill `practice_profile` only when the pages describe a specific management practice or a named agroforestry model. Copy species names, regions and rainfall figures exactly as printed. `stated_constraints` must capture every drawback, limitation, risk or trade-off the document mentions — these are as important as the benefits. For assessment or methodology pages, set practice_profile to null.

RULE 6 — SCOPE AND SIZE
Expected output per unit: practice or case study 5-15 passages, 2-10 claims, 2-8 links; agroforestry model 4-8 passages, 1-5 claims, 2-4 links; assessment excerpt 5-12 passages, 0-5 claims, 2-6 links. Prefer fewer, well-formed items over many fragments.
Use only the enum values provided by the schema. If no enum value fits, choose the closest and, for practices, use "other". Use "unstated" for climate zone or land use when the document does not say.
IDs: passage_id must be "{unit_id}__pNN" and claim_id "{unit_id}__cNN", numbered from 01 in document order.

RULE 7 — HONESTY
If the pages are partly unreadable, scanned, or cut off mid-section, set extraction_status "partial" and extract what is legible. Never fabricate content to fill the schema."""

def build_user_prompt(rec):
    return f"""Extract from this document unit.

unit_id: {rec['unit_id']}
source document: {rec['doc_id']}
unit title (from the table of contents): {rec['title']}
pages in this slice: {rec['n_pages']} (PDF pages {rec['pdf_start']}-{rec['pdf_end']}, printed page {rec['printed_start']} onward)

Use exactly this unit_id when constructing passage_id and claim_id values."""

In [ ]:
MAX_INLINE_MB = 15

def call_gemini(pdf_bytes, rec, model):
    part = types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf")
    cfg = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        response_mime_type="application/json",
        response_schema=UnitExtraction,
        temperature=0,
        max_output_tokens=32768,
    )
    return client.models.generate_content(
        model=model, contents=[part, build_user_prompt(rec)], config=cfg)

def extract_unit(rec, max_retries=3):
    pdf_path = ROOT / rec["path"]
    pdf_bytes = pdf_path.read_bytes()
    if len(pdf_bytes) > MAX_INLINE_MB * 1024 * 1024:
        raise RuntimeError(f"{rec['unit_id']} is {len(pdf_bytes)/1e6:.1f} MB, use the Files API")

    last_err = None
    for attempt in range(max_retries):
        model = MODEL if attempt < 2 else FALLBACK_MODEL
        try:
            resp = call_gemini(pdf_bytes, rec, model)
            data = json.loads(resp.text)
            data["_meta"] = {
                "unit_id": rec["unit_id"], "doc_id": rec["doc_id"],
                "model": model, "prompt_version": PROMPT_VERSION,
                "attempt": attempt + 1,
                "pdf_start": rec["pdf_start"], "pdf_end": rec["pdf_end"],
                "manifest_practices": rec.get("practices", []),
                "manifest_climate_zones": rec.get("climate_zones", []),
                "extracted_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "pdf_sha256": hashlib.sha256(pdf_bytes).hexdigest()[:16],
            }
            return data
        except Exception as e:
            last_err = e
            wait = min(30, (2 ** attempt) * 8) + random.uniform(0, 3)
            print(f"   attempt {attempt+1} failed ({type(e).__name__}), sleeping {wait:.0f}s")
            time.sleep(wait)
    raise last_err

In [ ]:
TIERS   = {"A"}          # start with A, then rerun with {"A","B"}
DOC_FILTER = None        # e.g. "fao2021_recarb_v3" to do one document first

todo = [r for r in manifest
        if r.get("tier","A") in TIERS
        and (DOC_FILTER is None or r["doc_id"] == DOC_FILTER)]
print(f"{len(todo)} units queued")

done = failed = skipped = 0
for i, rec in enumerate(todo, 1):
    out_path = OUT_DIR / rec["doc_id"] / f"{rec['unit_id']}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        skipped += 1
        continue

    print(f"[{i}/{len(todo)}] {rec['unit_id']} ({rec['n_pages']}p)", flush=True)
    try:
        data = extract_unit(rec)
        status = data.get("extraction_status", "ok")
        target = out_path if status != "no_content" else FAIL_DIR / f"{rec['unit_id']}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(data, indent=2, ensure_ascii=False))
        print(f"   {status}: {len(data['passages'])} passages, "
              f"{len(data['claims'])} claims, {len(data['links'])} links")
        done += 1
    except Exception as e:
        (FAIL_DIR / f"{rec['unit_id']}.error.txt").write_text(f"{type(e).__name__}: {e}")
        print(f"   FAILED: {type(e).__name__}: {str(e)[:150]}")
        failed += 1

    time.sleep(SLEEP_BETWEEN + random.uniform(0, 1))

print(f"\ndone {done} | failed {failed} | already existed {skipped}")

72 units queued
[17/72] srccl_ch3_desertification__w00 (12p)
   ok: 5 passages, 2 claims, 2 links
[18/72] srccl_ch3_desertification__w01 (12p)
   ok: 4 passages, 3 claims, 3 links
[19/72] srccl_ch3_desertification__w02 (12p)
   ok: 2 passages, 3 claims, 2 links
[20/72] srccl_ch3_desertification__w03 (12p)
   ok: 3 passages, 1 claims, 2 links
[21/72] srccl_ch3_desertification__w04 (12p)
   ok: 2 passages, 1 claims, 4 links
[22/72] srccl_ch3_desertification__w05 (12p)
   ok: 5 passages, 2 claims, 2 links
[23/72] srccl_ch3_desertification__w06 (12p)
   ok: 5 passages, 0 claims, 0 links
[24/72] srccl_ch3_desertification__w07 (12p)
   ok: 1 passages, 1 claims, 1 links
[25/72] srccl_ch3_desertification__w08 (10p)
   ok: 5 passages, 1 claims, 2 links
[26/72] srccl_ch4_land_degradation__w00 (12p)
   ok: 3 passages, 2 claims, 2 links
[27/72] srccl_ch4_land_degradation__w01 (12p)
   ok: 3 passages, 3 claims, 3 links
[28/72] srccl_ch4_land_degradation__w02 (12p)
   ok: 3 passages, 3 claims, 2 lin

In [ ]:
def norm(s):
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

rows, total_claims, unverified = [], 0, 0
for f in sorted(OUT_DIR.rglob("*.json")):
    if "_failures" in str(f):
        continue
    d = json.loads(f.read_text())
    uid = d["_meta"]["unit_id"]
    rec = next(m for m in manifest if m["unit_id"] == uid)
    raw = " ".join((p.extract_text() or "") for p in PdfReader(str(ROOT / rec["path"])).pages)
    raw_n = norm(raw)

    bad = [c for c in d["claims"] if norm(c["evidence_span"])[:90] not in raw_n]
    total_claims += len(d["claims"]); unverified += len(bad)
    rows.append((uid, d["extraction_status"], len(d["passages"]), len(d["claims"]),
                 len(bad), len(d["links"]),
                 sum(1 for l in d["links"] if l.get("condition"))))

print(f"{'unit':38} {'stat':8} {'pass':>4} {'clm':>4} {'unver':>5} {'link':>5} {'cond':>4}")
for r in rows:
    print(f"{r[0][:38]:38} {r[1]:8} {r[2]:>4} {r[3]:>4} {r[4]:>5} {r[5]:>5} {r[6]:>4}")

print(f"\nunits {len(rows)} | claims {total_claims} | "
      f"unverified spans {unverified} ({100*unverified/max(1,total_claims):.0f}%)")

unit                                   stat     pass  clm unver  link cond
v3_p01_cover_cropping                  ok         26   33     5    25   14
v3_p02_organic_mulch                   ok         22   27     1    29   10
v3_p03_crop_rotations                  ok         21   12     0    16    2
v3_p04_intercropping_multi             ok         22   16     0    12    9
v3_p05_intercropping_strip             ok         13   31     1    30    7
v3_p28_irrigation                      ok         35   17     1    15    8
v3_p38_agrisilvicultural               ok         38   24     2    24    6
v3_p39_silvopastoral                   ok         23   17     0    18    7
v3_p40_agrosilvopastoral               ok         22   20     0    18    7
v4_c01_notill_olive_lebanon            ok         17   16     2    15   11
v4_c05_ca_south_africa                 ok         24   25     1    17   12
v4_c06_intercropping_africa            ok         21   22     5    19    8
v4_c17_dehesa_iberia     

In [ ]:
import difflib, json, re
from pypdf import PdfReader

def canon(s):
    s = (s or "").lower()
    s = s.replace("\u00ad", "").replace("-\n", "").replace("ﬁ","fi").replace("ﬂ","fl")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

d = json.loads((OUT_DIR/"joshi2023_agj"/"joshi2023_agj__full.json").read_text())
rec = next(m for m in manifest if m["unit_id"]=="joshi2023_agj__full")
raw = canon(" ".join((p.extract_text() or "") for p in PdfReader(str(ROOT/rec["path"])).pages))

hard = 0
for c in d["claims"]:
    span = canon(c["evidence_span"])
    if span[:80] in raw:
        continue
    ratio = difflib.SequenceMatcher(None, span[:120], raw).quick_ratio()
    hard += 1
    print(f"[{ratio:.2f}] {c['evidence_span'][:130]}")
print("\nstill unmatched after normalisation:", hard, "of", len(d["claims"]))

[0.00] For example, cover crops in the moderate (3–7 Mg ha¯¹ per year-1) and high (>7 Mg ha¯¹ per year¯¹) biomass categories increased th
[0.00] For example, cover crops in the moderate (3–7 Mg ha¯¹ per year-1) and high (>7 Mg ha¯¹ per year¯¹) biomass categories increased th
[0.00] cover crop biomass <3 Mg ha¯¹ per year-¹ the cover crop did not increase percent SOC gains (95% CI, -0.5%-7.6%).
[0.00] The In(R) value ranged from negative to zero, which indicates a reduction to no change in SOC due to cover crop cultivation, where
[0.00] At an average SOC sequestration rate of 0.88 Mg ha¯¹ per year at 0-30 cm, current corn fields with cover crops are potentially seq
[0.00] At an average SOC sequestration rate of 0.88 Mg ha¯¹ per year at 0-30 cm, current corn fields with cover crops are potentially seq
[0.00] At an average SOC sequestration rate of 0.88 Mg ha¯¹ per year at 0-30 cm, current corn fields with cover crops are potentially seq

still unmatched after normalisation: 7 of 38


In [ ]:
for f in sorted((OUT_DIR/"handa2019_agroforestry_models").glob("*.json")):
    d = json.loads(f.read_text())
    pp = d.get("practice_profile")
    print(f"{f.stem:32} profile={'yes' if pp else 'NO'} "
          f"species={len(pp['tree_species']) if pp else 0} "
          f"constraints={len(pp['stated_constraints']) if pp else 0} "
          f"rain={pp.get('rainfall_mm_low') if pp else None}")

handa_2_1_bakain_agrisilvi       profile=yes species=1 constraints=1 rain=None
handa_3_1_melia_dubia            profile=yes species=1 constraints=1 rain=400
handa_4_1_ailanthus_rainfed      profile=yes species=1 constraints=4 rain=None
handa_4_2_shisham                profile=yes species=1 constraints=5 rain=None
handa_4_3_aonla_agrihorti        profile=yes species=1 constraints=2 rain=None
handa_5_1_melia_azedarach        profile=yes species=1 constraints=2 rain=None


In [ ]:
def canon(s):
    s = (s or "").lower().replace("\u00ad","").replace("-\n","")
    s = s.replace("ﬁ","fi").replace("ﬂ","fl")
    s = re.sub(r"[¯⁻−–—]", "-", s)
    s = re.sub(r"[⁰¹²³⁴⁵⁶⁷⁸⁹]", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

In [ ]:
import json
AER_RAIN = {"handa_3_1_melia_dubia": (400, 500), "handa_6_1_three_tier_paddy": (600, 1000),
            "handa_6_2_teak_agrisilvi": (600, 1000), "handa_6_3_sapota_teak": (600, 1000)}
for f in (OUT_DIR/"handa2019_agroforestry_models").glob("*.json"):
    d = json.loads(f.read_text()); pp = d.get("practice_profile")
    if pp and f.stem in AER_RAIN and pp.get("rainfall_mm_low") is None:
        pp["rainfall_mm_low"], pp["rainfall_mm_high"] = AER_RAIN[f.stem]
        pp["_rainfall_source"] = "AER heading via manifest"
        f.write_text(json.dumps(d, indent=2, ensure_ascii=False))
        print("patched", f.stem)

In [ ]:
ASSESSMENT_PROMPT = """You are a scientific data extraction system for an environmental knowledge base. You extract what a document states. You never infer, estimate, generalise, or use knowledge from outside the provided pages.

These pages come from a global scientific assessment (IPCC or IPBES). Such assessments state findings as synthesised conclusions with confidence language and dense inline references. Extract accordingly.

TASK
Read the attached PDF pages and return a single JSON object matching the provided schema.

RULE 1 — PASSAGES ARE VERBATIM, BUT CITATIONS MAY BE STRIPPED
`text` must otherwise be copied character-for-character from the document. The ONLY permitted edit is removing inline reference clutter: parenthetical author-year citations such as "(Smith et al., 2015; Jones, 2017)", curly-brace cross-references such as "{4.2.1}", and bracketed numeric references. Remove the citation, keep every other word, and close the spacing naturally. Never paraphrase, summarise, merge distant sentences, or reword.
KEEP confidence language intact: "high confidence", "medium agreement", "likely", "robust evidence" are part of the finding.
Set `page_start`/`page_end` to the PRINTED page numbers shown on the page.

RULE 2 — TARGET DENSITY
These pages are information-dense. Expect 10-20 passages per 12-page unit, each typically 2-6 sentences. Under-extraction is the main failure mode: if you produce fewer than 8 passages for a full unit, you have skipped usable content. Prefer several focused passages over one long one.

RULE 3 — WHAT TO PRIORITISE, IN ORDER
1. Statements linking TWO OR MORE environmental variables (e.g. soil carbon and biodiversity, water availability and species survival, land use and habitat fragmentation). These are the highest value content in this document.
2. Statements about management practices, restoration options, or responses, and their effects.
3. Statements about drylands, arid or semi-arid regions, or India and South Asia specifically.
4. Quantified findings of any kind.
5. Definitions of key terms (degradation, desertification, restoration) where stated formally.

RULE 4 — WHAT TO SKIP
Do not create passages from: reference lists, author or contributor lists, acknowledgements, chapter tables of contents, executive-summary bullet lists that merely repeat body text verbatim, figure and table captions with no standalone finding, cross-reference indexes, page headers and footers, or pages that are only figures. If a full unit contains none of the priority content in Rule 3, return extraction_status "no_content".

RULE 5 — CLAIMS NEED RECEIPTS
Create a claim wherever the document states an effect on a metric, whether or not a number is given. A directional finding with confidence language but no number is still a valid claim: set direction and leave value fields null.
`evidence_span` must be the exact sentence or clause from the document, copied verbatim INCLUDING any citations it contains (do not strip citations inside evidence_span — only inside passage text).
- `confidence_in_extraction`: "explicit" when the text states the value directly; "derived" when read from a figure or table.
- Record `conditions` exactly as stated (region, climate, land use, timescale). If unstated, null.
- Put the assessment's own confidence wording into `basis` when present, e.g. "high confidence".

RULE 6 — LINKS ARE THE MAIN OUTPUT OF THIS DOCUMENT TYPE
Assessments exist to state relationships between variables. Extract every cause-effect relationship the document asserts, including metric-to-metric links with no practice involved (e.g. soil organic carbon to microbial diversity, water availability to species richness, habitat fragmentation to pollinator abundance). Expect 6-15 links per unit.
The `condition` field is critical: where the document says an effect depends on climate, region, soil, land use or timescale, record it in short form (e.g. "drylands", "arid|semi-arid", "sandy soils", "decadal timescale"). Use null only when the document states the relationship unconditionally.
Use `strength_stated` to carry the assessment's confidence: "strong" for high confidence or robust evidence, "moderate" for medium, "weak" for low confidence or limited evidence, "unstated" otherwise.

RULE 7 — NO PRACTICE PROFILE
Set practice_profile to null. These pages describe findings, not a single named management practice.

RULE 8 — IDS AND HONESTY
passage_id must be "{unit_id}__pNN" and claim_id "{unit_id}__cNN", numbered from 01 in document order. Use only schema enum values; "unstated" where the document does not say.
If pages are partly unreadable or cut mid-section, set extraction_status "partial" and extract what is legible. Never fabricate content to fill the schema."""

In [ ]:
ASSESSMENT_DOCS = {"ipcc2019_srccl", "ipbes2018_ldr"}

def call_gemini(pdf_bytes, rec, model):
    prompt = ASSESSMENT_PROMPT if rec["doc_id"] in ASSESSMENT_DOCS else SYSTEM_PROMPT
    part = types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf")
    cfg = types.GenerateContentConfig(
        system_instruction=prompt,
        response_mime_type="application/json",
        response_schema=UnitExtraction,
        temperature=0,
        max_output_tokens=32768,
    )
    return client.models.generate_content(
        model=model, contents=[part, build_user_prompt(rec)], config=cfg)

PROMPT_VERSION = "v1.1-assessment"

# archive the thin v1.0 outputs instead of deleting them
import shutil
ARCH = OUT_DIR / "_v10_assessment_archive"
for doc_id in ASSESSMENT_DOCS:
    src = OUT_DIR / doc_id
    if src.exists():
        dst = ARCH / doc_id
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        print("archived", doc_id)

archived ipbes2018_ldr
archived ipcc2019_srccl


In [ ]:
TIERS = {"A"}
DOC_FILTER = None
todo = [r for r in manifest
        if r.get("tier","A") in TIERS and r["doc_id"] in ASSESSMENT_DOCS]

In [ ]:
ASSESSMENT_DOCS = {"ipcc2019_srccl", "ipbes2018_ldr"}

todo = [r for r in manifest
        if r.get("tier", "A") == "A" and r["doc_id"] in ASSESSMENT_DOCS]
print(f"{len(todo)} assessment units queued")

done = failed = skipped = 0
for i, rec in enumerate(todo, 1):
    out_path = OUT_DIR / rec["doc_id"] / f"{rec['unit_id']}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        skipped += 1
        continue

    print(f"[{i}/{len(todo)}] {rec['unit_id']} ({rec['n_pages']}p)", flush=True)
    try:
        data = extract_unit(rec)
        status = data.get("extraction_status", "ok")
        target = out_path if status != "no_content" else FAIL_DIR / f"{rec['unit_id']}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(data, indent=2, ensure_ascii=False))
        n_cond = sum(1 for l in data["links"] if l.get("condition"))
        print(f"   {status}: {len(data['passages'])} passages, "
              f"{len(data['claims'])} claims, {len(data['links'])} links ({n_cond} conditional)")
        done += 1
    except Exception as e:
        (FAIL_DIR / f"{rec['unit_id']}.error.txt").write_text(f"{type(e).__name__}: {e}")
        print(f"   FAILED: {type(e).__name__}: {str(e)[:150]}")
        failed += 1

    time.sleep(SLEEP_BETWEEN + random.uniform(0, 1.5))

print(f"\ndone {done} | failed {failed} | existed {skipped}")

50 assessment units queued
[1/50] srccl_ch3_desertification__w00 (12p)
   ok: 8 passages, 4 claims, 4 links (0 conditional)
[2/50] srccl_ch3_desertification__w01 (12p)
   ok: 10 passages, 8 claims, 8 links (8 conditional)
[3/50] srccl_ch3_desertification__w02 (12p)
   ok: 10 passages, 5 claims, 9 links (0 conditional)
[4/50] srccl_ch3_desertification__w03 (12p)
   ok: 9 passages, 6 claims, 8 links (2 conditional)
[5/50] srccl_ch3_desertification__w04 (12p)
   ok: 8 passages, 8 claims, 7 links (0 conditional)
[6/50] srccl_ch3_desertification__w05 (12p)
   ok: 5 passages, 3 claims, 3 links (0 conditional)
[7/50] srccl_ch3_desertification__w06 (12p)
   ok: 8 passages, 2 claims, 2 links (0 conditional)
[8/50] srccl_ch3_desertification__w07 (12p)
   ok: 8 passages, 7 claims, 6 links (5 conditional)
[9/50] srccl_ch3_desertification__w08 (10p)
   ok: 8 passages, 3 claims, 3 links (3 conditional)
[10/50] srccl_ch4_land_degradation__w00 (12p)
   ok: 15 passages, 3 claims, 6 links (2 conditional

In [ ]:
import re, json

def canon(s):
    s = (s or "").lower().replace("\u00ad", "").replace("-\n", "")
    s = s.replace("ﬁ", "fi").replace("ﬂ", "fl")
    s = re.sub(r"[¯⁻−–—]", "-", s)
    s = re.sub(r"[⁰¹²³⁴⁵⁶⁷⁸⁹]", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def stats(folder):
    p = c = l = cond = n = 0
    for f in folder.rglob("*.json"):
        d = json.loads(f.read_text())
        if "_meta" not in d:
            continue
        n += 1; p += len(d["passages"]); c += len(d["claims"]); l += len(d["links"])
        cond += sum(1 for x in d["links"] if x.get("condition"))
    return n, p, c, l, cond

print(f"{'':28} {'units':>5} {'pass':>5} {'clm':>5} {'link':>5} {'cond':>5} {'p/unit':>7}")
for label, folder in [("v1.0 archived", OUT_DIR / "_v10_assessment_archive"),
                      ("v1.1 new", None)]:
    if folder is None:
        n = p = c = l = cond = 0
        for d_id in ASSESSMENT_DOCS:
            s = stats(OUT_DIR / d_id)
            n, p, c, l, cond = n + s[0], p + s[1], c + s[2], l + s[3], cond + s[4]
    else:
        n, p, c, l, cond = stats(folder)
    print(f"{label:28} {n:>5} {p:>5} {c:>5} {l:>5} {cond:>5} {p/max(n,1):>7.1f}")

# per-unit detail for the new run
print()
for d_id in sorted(ASSESSMENT_DOCS):
    for f in sorted((OUT_DIR / d_id).glob("*.json")):
        d = json.loads(f.read_text())
        raw = " ".join((pg.extract_text() or "")
                       for pg in PdfReader(str(ROOT / next(
                           m["path"] for m in manifest if m["unit_id"] == d["_meta"]["unit_id"]))).pages)
        raw_n = canon(raw)
        bad = sum(1 for x in d["claims"] if canon(x["evidence_span"])[:80] not in raw_n)
        cond = sum(1 for x in d["links"] if x.get("condition"))
        print(f"{d['_meta']['unit_id'][:36]:36} pass {len(d['passages']):>3}  "
              f"clm {len(d['claims']):>3}  unver {bad:>2}  link {len(d['links']):>3}  cond {cond:>3}")

                             units  pass   clm  link  cond  p/unit
v1.0 archived                   48   145    94    99    19     3.0
v1.1 new                        50   402   205   238    62     8.0

ipbes_ch4_status_trends__w00         pass   7  clm   2  unver  0  link   6  cond   0
ipbes_ch4_status_trends__w01         pass   9  clm   2  unver  0  link   5  cond   3
ipbes_ch4_status_trends__w02         pass  14  clm   7  unver  0  link   7  cond   1
ipbes_ch4_status_trends__w03         pass   9  clm   6  unver  0  link   7  cond   2
ipbes_ch4_status_trends__w04         pass   8  clm   4  unver  0  link   7  cond   3
ipbes_ch4_status_trends__w05         pass   9  clm   5  unver  0  link   7  cond   0
ipbes_ch4_status_trends__w06         pass   6  clm   4  unver  0  link   7  cond   3
ipbes_ch4_status_trends__w07         pass   6  clm   2  unver  0  link   2  cond   0
ipbes_ch4_status_trends__w08         pass   8  clm   3  unver  0  link   3  cond   2
ipbes_ch4_status_trends__w09     

### P1

In [ ]:
!pip install pymupdf -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 33.6 MB/s eta 0:00:00


In [ ]:
import fitz
for f in ["fsi2023_isfr_v1.pdf", "fsi2023_isfr_v2.pdf"]:
    doc = fitz.open(ROOT / "p1" / f)
    print(f, len(doc), "pages")
    for p in ([240, 250, 262, 275] if "v1" in f else [275, 285, 290]):
        if p < len(doc):
            page = doc[p - 1]
            print(f"  p{p}: {len(page.get_text().strip()):>6} chars, {len(page.get_images()):>2} images")

fsi2023_isfr_v1.pdf 368 pages
  p240:   2838 chars,  0 images
  p250:     21 chars,  1 images
  p262:   1364 chars,  1 images
  p275:   1656 chars,  1 images
fsi2023_isfr_v2.pdf 428 pages
  p275:    120 chars,  4 images
  p285:   1737 chars,  1 images
  p290:    340 chars,  2 images


In [ ]:
INVENTORY_PROMPT = """You are a scientific data extraction system for an environmental knowledge base. You extract what a document states. You never infer, estimate, generalise, or use knowledge from outside the provided pages.

These pages come from a national statistical inventory or a technical methodology manual (Forest Survey of India report, or an FAO methodology document). Such documents report measured area statistics, inventory results, definitions and procedures. They do NOT establish cause-and-effect relationships between management practices and environmental outcomes.

TASK
Read the attached PDF pages and return a single JSON object matching the provided schema.

RULE 1 — NO CLAIMS, NO LINKS
Return `claims` as an empty list and `links` as an empty list, always, without exception. An area statistic ("forest cover increased by 1,445 square kilometres"), an inventory total, or a methodological step is NOT a claim about a practice affecting a metric, and must not be recorded as one. This document type contributes baseline context and definitions only. Do not attempt to convert statistics into effects.

RULE 2 — PASSAGES ARE VERBATIM
`text` must be copied character-for-character from the document. You may remove inline reference clutter (parenthetical author-year citations, bracketed numeric references) and nothing else. Never paraphrase or reword.
Set `page_start`/`page_end` to the PRINTED page numbers shown on the page.

RULE 3 — READ TABLES, CHARTS AND MAPS AS CONTENT
Many pages are dominated by tables, bar charts, pie charts or maps with little surrounding prose. These carry the document's main information and must be extracted, not skipped.
- For a TABLE: create one passage with content_role "table". Reproduce the table's caption verbatim, then render the rows as readable lines, preserving every label, figure and unit exactly as printed (e.g. "Forests | 3,688.22 | 22.63"). Keep the source note if one is printed beneath.
- For a CHART or MAP: create one passage with content_role "table". Reproduce the caption and axis or legend labels verbatim, then state the values that are explicitly labelled on the figure. If values are not printed on the figure, describe only what the labels and legend state and record no numbers. Never estimate a value from a bar height, a line position, or a map colour.
- If a page is a decorative photograph with no data, skip it.

RULE 4 — WHAT TO PRIORITISE
1. Land use and land cover statistics: area under forest, tree cover, agroforestry, pastures and grazing land, culturable wasteland, fallow land, net area sown.
2. Agroforestry extent, growing stock and change over time.
3. Soil and site conditions recorded by inventory: soil depth, humus, soil organic carbon, presence of grasses and undergrowth.
4. Disturbance and human impact records: erosion, grazing incidence, invasive species, illicit felling, lopping, fire incidence.
5. Regeneration status, canopy cover, crop composition.
6. Formal definitions of technical terms (forest cover, tree cover, recorded forest area, trees outside forests, soil organic carbon stock, sequestration rate, reference depth).
7. Units, conversion factors and measurement conventions.

RULE 5 — WHAT TO SKIP
Do not create passages from: forewords, prefaces, messages from officials, acknowledgements, tables of contents, lists of abbreviations, satellite sensor specifications, software and processing workflows, sampling design mathematics, annexes, contributor lists, references, and page headers or footers.

RULE 6 — GEOGRAPHIC AND TEMPORAL LABELS
Always record in the passage `topic` field which state, district, region or assessment year the content refers to, when the page states it. These pages are only useful later if their geography and date are attached.

RULE 7 — SIZE AND SCOPE
Expect 6-14 passages per unit; a table-dense unit may reach 20. Use only schema enum values, and "unstated" where the document does not say. Set practice_profile to null.
IDs: passage_id must be "{unit_id}__pNN", numbered from 01 in document order.
If pages are unreadable or contain none of the priority content, set extraction_status "no_content" with empty lists. Never fabricate content to fill the schema."""

In [ ]:
import json
from pathlib import Path
from pypdf import PdfReader, PdfWriter

P1_UNITS_DIR = ROOT / "p1_units"

P1 = {
 "tewari_agroforestry_hot_arid": {"path": "p1/tewari_agroforestry_hot_arid.pdf", "offset": 0,
   "prompt": "practice", "split": "unit",
   "units": [{"unit_id": "tewari_hot_arid__full", "title": "Agroforestry in hot arid environments", "s": 1, "e": 26,
              "climate_zones": ["arid"], "practices": ["agroforestry_agrisilvicultural"]}]},

 "keerthika2026_agroforestry_semiarid": {"path": "p1/keerthika2026_agroforestry_semiarid.pdf", "offset": 0,
   "prompt": "practice", "split": "unit",
   "units": [{"unit_id": "keerthika2026__full", "title": "Multitier agroforestry under deficit irrigation, semi-arid Rajasthan", "s": 1, "e": 14,
              "climate_zones": ["semi-arid"], "practices": ["agroforestry_agrisilvicultural", "irrigation"]}]},

 "ipbes2016_pollinators": {"path": "p1/ipbes2016_pollinators.pdf", "offset": 0,
   "prompt": "assessment", "split": "window", "window": 12, "overlap": 1,
   "units": [{"unit_id": "ipbes_pollinators", "title": "Pollinators, pollination and food production", "s": 1, "e": 40}]},

 "fao2019_sowbfa": {"path": "p1/fao2019_sowbfa.pdf", "offset": 44,
   "prompt": "assessment", "split": "window", "window": 12, "overlap": 1,
   "units": [
     {"unit_id": "sowbfa_pollination",   "title": "4.3.4 Associated biodiversity for pollination",      "s": 173, "e": 179},
     {"unit_id": "sowbfa_soil_services", "title": "4.3.6 Soil-related ecosystem services",              "s": 184, "e": 193},
     {"unit_id": "sowbfa_water_services","title": "4.3.7 Water-related ecosystem services",             "s": 192, "e": 198},
     {"unit_id": "sowbfa_habitat",       "title": "4.3.9 Habitat provisioning",                         "s": 198, "e": 202},
     {"unit_id": "sowbfa_diversification","title": "5.5 Diversification in production systems",         "s": 267, "e": 293},
     {"unit_id": "sowbfa_soil_biodiv_mgmt","title": "5.6.3 Practices to enhance soil biodiversity",     "s": 297, "e": 301},
   ]},

 "fao2018_soil_pollution": {"path": "p1/fao2018_soil_pollution.pdf", "offset": 12,
   "prompt": "assessment", "split": "unit",
   "units": [
     {"unit_id": "soilpol_agri_sources", "title": "1.3.2.6 Agricultural and livestock sources",         "s": 27, "e": 32},
     {"unit_id": "soilpol_np_pesticides","title": "1.4.2-1.4.3 Nitrogen, phosphorus and pesticides",    "s": 32, "e": 39},
     {"unit_id": "soilpol_ecosystem",    "title": "2.2 Impacts on ecosystem services from agriculture", "s": 63, "e": 69},
     {"unit_id": "soilpol_agronomic_fix","title": "3.3 Agronomic practices to minimise contamination",  "s": 92, "e": 96},
   ]},

 "fsi2023_isfr_v1": {"path": "p1/fsi2023_isfr_v1.pdf", "offset": 46,
   "prompt": "inventory", "split": "window", "window": 10, "overlap": 1,
   "units": [
     {"unit_id": "isfr1_agroforestry", "title": "Ch7 Trees in agroforestry systems", "s": 237, "e": 254},
     {"unit_id": "isfr1_soil_disturb", "title": "8.4.1-8.4.2 Enabling conditions and disturbance", "s": 258, "e": 283},
   ]},

 "fsi2023_isfr_v2": {"path": "p1/fsi2023_isfr_v2.pdf", "offset": 36,
   "prompt": "inventory", "split": "unit",
   "units": [
     {"unit_id": "isfr2_punjab",    "title": "Punjab forest and tree resources",    "s": 270, "e": 282, "region": "Punjab"},
     {"unit_id": "isfr2_rajasthan", "title": "Rajasthan forest and tree resources", "s": 280, "e": 293, "region": "Rajasthan"},
   ]},

 "fao_gsocseq_intro": {"path": "p1/fao_gsocseq_intro.pdf", "offset": 0,
   "prompt": "inventory", "split": "window", "window": 12, "overlap": 1,
   "units": [{"unit_id": "gsocseq_intro", "title": "GSOCseq introduction and methodology", "s": 1, "e": 85}]},
}

def windows(s, e, size, ov):
    while s <= e:
        end = min(s + size - 1, e)
        yield s, end
        if end == e: break
        s = end - ov + 1

p1_manifest, made, skipped = [], 0, 0
for doc_id, doc in P1.items():
    src = ROOT / doc["path"]
    if not src.exists():
        print("MISSING:", src); continue
    reader = PdfReader(str(src)); n = len(reader.pages)
    dest_dir = P1_UNITS_DIR / doc_id; dest_dir.mkdir(parents=True, exist_ok=True)

    for u in doc["units"]:
        e_cap = min(u["e"], n)
        pieces = ([(u["s"], e_cap)] if doc["split"] != "window"
                  else list(windows(u["s"], e_cap, doc["window"], doc["overlap"])))
        for k, (s, e) in enumerate(pieces):
            uid = u["unit_id"] if len(pieces) == 1 else f"{u['unit_id']}__w{k:02d}"
            dest = dest_dir / f"{uid}.pdf"
            if dest.exists():
                skipped += 1
            else:
                w = PdfWriter()
                for i in range(s - 1, e): w.add_page(reader.pages[i])
                with open(dest, "wb") as f: w.write(f)
                made += 1
            p1_manifest.append({
                "unit_id": uid, "doc_id": doc_id, "parent_unit": u["unit_id"], "title": u["title"],
                "pdf_start": s, "pdf_end": e, "n_pages": e - s + 1,
                "printed_start": s - doc["offset"], "path": str(dest.relative_to(ROOT)),
                "prompt": doc["prompt"], "practices": u.get("practices", []),
                "climate_zones": u.get("climate_zones", []), "region": u.get("region"), "tier": "A",
            })

(ROOT / "p1_units_manifest.json").write_text(json.dumps(p1_manifest, indent=2))
print(f"created {made}, existed {skipped}, total P1 units {len(p1_manifest)}")
for r in p1_manifest[:5]:
    print(" ", r["unit_id"], r["pdf_start"], "-", r["pdf_end"], f"({r['prompt']})")

created 33, existed 0, total P1 units 33
  tewari_hot_arid__full 1 - 26 (practice)
  keerthika2026__full 1 - 14 (practice)
  ipbes_pollinators__w00 1 - 12 (assessment)
  ipbes_pollinators__w01 12 - 23 (assessment)
  ipbes_pollinators__w02 23 - 34 (assessment)


In [ ]:
PROMPT_MAP = {"practice": SYSTEM_PROMPT, "assessment": ASSESSMENT_PROMPT, "inventory": INVENTORY_PROMPT}
PROMPT_VERSION = "v1.2-p1"

def call_gemini(pdf_bytes, rec, model):
    prompt = PROMPT_MAP[rec.get("prompt", "practice")]
    part = types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf")
    cfg = types.GenerateContentConfig(
        system_instruction=prompt, response_mime_type="application/json",
        response_schema=UnitExtraction, temperature=0, max_output_tokens=32768)
    return client.models.generate_content(model=model, contents=[part, build_user_prompt(rec)], config=cfg)

todo = json.loads((ROOT / "p1_units_manifest.json").read_text())
print(f"{len(todo)} P1 units queued\n")

done = failed = skipped = 0
for i, rec in enumerate(todo, 1):
    out_path = OUT_DIR / rec["doc_id"] / f"{rec['unit_id']}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        skipped += 1; continue
    print(f"[{i}/{len(todo)}] {rec['unit_id']} ({rec['n_pages']}p, {rec['prompt']})", flush=True)
    try:
        data = extract_unit(rec)
        data["_meta"]["prompt_type"] = rec["prompt"]
        status = data.get("extraction_status", "ok")
        target = out_path if status != "no_content" else FAIL_DIR / f"{rec['unit_id']}.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(data, indent=2, ensure_ascii=False))
        print(f"   {status}: {len(data['passages'])} pass, {len(data['claims'])} clm, {len(data['links'])} link")
        done += 1
    except Exception as e:
        (FAIL_DIR / f"{rec['unit_id']}.error.txt").write_text(f"{type(e).__name__}: {e}")
        print(f"   FAILED: {type(e).__name__}: {str(e)[:150]}")
        failed += 1
    time.sleep(SLEEP_BETWEEN + random.uniform(0, 1.5))

print(f"\ndone {done} | failed {failed} | existed {skipped}")

33 P1 units queued

[1/33] tewari_hot_arid__full (26p, practice)
   ok: 3 pass, 2 clm, 2 link
[2/33] keerthika2026__full (14p, practice)
   ok: 4 pass, 2 clm, 2 link
[3/33] ipbes_pollinators__w00 (12p, assessment)
   ok: 8 pass, 4 clm, 5 link
[4/33] ipbes_pollinators__w01 (12p, assessment)
   ok: 7 pass, 5 clm, 6 link
[5/33] ipbes_pollinators__w02 (12p, assessment)
   ok: 8 pass, 6 clm, 4 link
[6/33] ipbes_pollinators__w03 (7p, assessment)
   ok: 5 pass, 2 clm, 2 link
[7/33] sowbfa_pollination (7p, assessment)
   ok: 8 pass, 3 clm, 4 link
[8/33] sowbfa_soil_services (10p, assessment)
   ok: 8 pass, 6 clm, 6 link
[9/33] sowbfa_water_services (7p, assessment)
   ok: 8 pass, 6 clm, 6 link
[10/33] sowbfa_habitat (5p, assessment)
   ok: 8 pass, 4 clm, 3 link
[11/33] sowbfa_diversification__w00 (12p, assessment)
   ok: 7 pass, 5 clm, 7 link
[12/33] sowbfa_diversification__w01 (12p, assessment)
   ok: 6 pass, 4 clm, 4 link
[13/33] sowbfa_diversification__w02 (5p, assessment)
   attempt 1 fail

In [ ]:
import shutil
for d in ["tewari_agroforestry_hot_arid", "keerthika2026_agroforestry_semiarid"]:
    shutil.rmtree(OUT_DIR / d, ignore_errors=True)

todo2 = [r for r in json.loads((ROOT / "p1_units_manifest.json").read_text())
         if r["doc_id"] in {"tewari_agroforestry_hot_arid", "keerthika2026_agroforestry_semiarid"}]
for r in todo2:
    r["prompt"] = "assessment"

for rec in todo2:
    out = OUT_DIR / rec["doc_id"] / f"{rec['unit_id']}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    data = extract_unit(rec)
    data["_meta"]["prompt_type"] = "assessment"
    out.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    print(rec["unit_id"], "->", len(data["passages"]), "pass,", len(data["claims"]), "clm,", len(data["links"]), "link")
    time.sleep(SLEEP_BETWEEN)

tewari_hot_arid__full -> 6 pass, 6 clm, 6 link
keerthika2026__full -> 7 pass, 5 clm, 4 link


In [ ]:
d = json.loads((OUT_DIR / "fao2019_sowbfa" / "sowbfa_diversification__w02.json").read_text())
for p in d["passages"][:3]:
    print(p["content_role"], "|", p["text"][:160].replace("\n", " "), "\n")

description | Semi-intensive practices involve the addition of manure to promote growth of phytoplankton, but do not involve adding feed supplements, or only in very limited  

description | Intensive systems provide the highest levels of output, but are also the most expensive to operate. They involve the use of high-quality pellet feed that covers 

description | Another way of classifying polyculture systems is on the basis of spatial organization. In this respect, the three main types are direct, cage-cum-pond and sequ 



In [ ]:
(OUT_DIR / "fao2019_sowbfa" / "sowbfa_diversification__w02.json").unlink()
print("deleted aquaculture unit")

deleted aquaculture unit


In [ ]:
d = json.loads((OUT_DIR / "keerthika2026_agroforestry_semiarid" / "keerthika2026__full.json").read_text())
for c in d["claims"]:
    print(f"{c['metric']:18} {c['direction']:9} {c['value']} {c['unit'] or ''} | {c['conditions']}")

crop_yield         increase  43.33 percent_change | 1 CPE vs 0.6 CPE
crop_yield         increase  8.15 percent_change | 0.6 CPE vs 0.8 CPE
biomass            increase  41 percent_change | 1 CPE vs 0.6 CPE
nitrogen           decrease  28.75 percent_change | compared to initial soil values
nutrient_availability increase  8.76 percent_change | compared to initial soil values


In [ ]:
import json, random
from collections import Counter
links, nodes = [], Counter()
for f in OUT_DIR.rglob("*.json"):
    if "_v10" in str(f) or "_failures" in str(f): continue
    for l in json.loads(f.read_text()).get("links", []):
        links.append(l); nodes[l["from_node"]] += 1; nodes[l["to_node"]] += 1
print(len(links), "links,", len(nodes), "distinct nodes\n")
for n, c in nodes.most_common(25): print(f"{c:>4}  {n}")

646 links, 353 distinct nodes

 110  SOC
  82  crop_yield
  60  erosion
  60  intercropping
  57  cover_crops
  49  agroforestry_agrisilvicultural
  38  nutrient_availability
  37  soil_moisture
  27  nitrogen
  23  water_holding_capacity
  23  temperature
  23  carbon_sequestration_rate
  21  soil_biota
  21  reduced_tillage
  21  species_richness
  20  biomass
  18  mulching
  18  agroforestry_agrosilvopastoral
  17  bulk_density
  17  no_till
  16  pollution_load
  16  crop_rotation
  16  soil_organic_matter
  16  agroforestry_silvopastoral
  11  irrigation


In [ ]:
tail = [n for n, c in nodes.items() if c == 1]
print(len(tail), "singleton nodes\n")
for n in sorted(tail)[:60]:
    print(" ", n)

253 singleton nodes

  BCA abundance and species richness
  Biochar
  CO2 and N2O emissions
  CO2 fertilisation
  CO2 fertilization
  Charcoal production
  Closing yield gaps
  Conflict
  Costa Rican programme
  European farm subsidies
  FSC certification
  Fire frequency
  GFGP plots
  Ground cover < 50%
  Human Development Index
  Increasing soil organic carbon
  Integrated water management
  Livelihood sensitivity
  Migration
  Mongolian pine plantations
  N2O flux
  Napier grass
  No-till management
  Planting of non-native species
  REDD+
  SLM practices
  Terracing
  Tillage and harvesting
  Wild animal management
  addition of topsoil and composts
  afforestation/reforestation on degraded lands
  afforestation/reforestation on native grasslands
  agri-silvi system
  agricultural intensification
  agricultural productivity
  agriculture
  agro-biodiversity
  agroforestry_horti_pastoral
  animal or green manures
  application of compost
  aquaponics
  aridity
  atmospheric CO2
  a

In [ ]:
for n in sorted(tail)[60:]:
    print(" ", n)

  conversion of grassland to cropland
  copper immobilization
  cover crops
  cover on soil surface
  crop pollination
  crop residue mulch
  crop residue mulches
  crop residues and by-products
  crop rotations with forage legumes
  crop yield stability
  damage on local communities
  decomposition of legume residues
  defaunation
  deforestation
  deforestation and forest degradation
  deforestation and widespread degradation
  diversification practices
  diversified farming systems
  drip irrigation
  earthworm functional diversity
  ecosystem diversity
  ecosystem multifunctionality
  ecosystem services
  enhanced_mineral_weathering
  environmental services
  erosion factors
  excess N fertilization
  excess irrigation
  exclosure land management
  exclosures
  fertilizer addition
  fertilizer application
  fertilizer use
  fire intensity
  flood irrigation
  flower strips
  food waste reduction
  fragmentation
  fuelwood harvesting
  full irrigation
  funding
  goat stocking densi

In [ ]:
import json, re
from collections import Counter, defaultdict

# ── 1. canonical nodes promoted from the singleton list ──────────────
PROMOTED = {
 "soil_erosion","runoff","soil_fertility","nutrient_cycling","water_availability",
 "fragmentation","invasive_species","grazing_pressure","fire_frequency","tillage",
 "vegetation_cover","soil_compaction","tree_cover","deforestation","afforestation",
 "land_conversion","urbanization","pollination_services","pest_control_services",
 "ghg_emissions","N2O_emissions","nitrogen_loss","salinity","waterlogging",
 "ecosystem_multifunctionality","resilience","biochar","green_manure","fertilizer_use",
 "pesticide_use","restoration","ground_cover","soil_crusts","plant_diversity",
 "beneficial_microbes","earthworm_diversity","atmospheric_CO2","net_primary_production",
}

# ── 2. exact aliases: variant (lowercased) -> canonical ──────────────
ALIASES = {
 # SOC family
 "soil carbon stocks":"SOC", "carbon sequestration":"SOC", "soil organic matter":"SOC",
 "organic matter":"SOC", "increasing soil organic carbon":"SOC",
 "mitigation (increasing carbon stores)":"SOC", "root-associated carbon inputs":"SOC",
 "belowground carbon cycling":"SOC", "soil carbon":"SOC",
 # erosion / water movement
 "erosion":"soil_erosion", "soil losses":"soil_erosion", "erosion factors":"soil_erosion",
 "water erosion":"soil_erosion", "wind erosion":"soil_erosion", "sediment yield":"soil_erosion",
 "surface water runoff":"runoff", "soil water storage":"water_holding_capacity",
 "low soil water availability":"water_availability", "water scarcity":"water_availability",
 "moisture":"soil_moisture", "irrigation water":"water_availability",
 # practices
 "cover crops":"cover_crops", "legume residues":"cover_crops",
 "decomposition of legume residues":"cover_crops", "green manure":"green_manure",
 "animal or green manures":"green_manure", "terracing":"terracing",
 "no-till management":"no_till", "minimum_tillage":"reduced_tillage",
 "conservation tillage and crop rotation":"conservation_agriculture",
 "conservation agriculture with mulch":"conservation_agriculture",
 "slm practices":"conservation_agriculture", "conservation practices":"conservation_agriculture",
 "soil and water conservation measures":"conservation_agriculture",
 "soil conservation measures":"conservation_agriculture",
 "stone bunds and trenches":"terracing", "agri-silvi system":"agroforestry_agrisilvicultural",
 "parkland agroforestry":"agroforestry_agrisilvicultural",
 "tree-based agroecosystems":"agroforestry_agrisilvicultural",
 "home gardens":"agroforestry_agrisilvicultural", "shelterbelts":"shelterbelts",
 "wind strip cropping":"shelterbelts", "flower strips":"hedges_buffer_strips",
 "exclosures":"grazing_exclusion", "exclosure land management":"grazing_exclusion",
 "sustainable grazing":"rotational_grazing", "improved grazing land management":"rotational_grazing",
 "constant heavy grazing":"grazing_pressure", "goat stocking density":"grazing_pressure",
 "livestock numbers":"grazing_pressure", "pasture intensification":"grazing_pressure",
 "integrated crop, livestock and forestry systems":"crop_livestock_integration",
 "diversified farming systems":"crop_diversification",
 "mixed and diversified cropping systems":"crop_diversification",
 "diversification practices":"crop_diversification",
 "crop rotations with forage legumes":"crop_rotation", "crop rotations":"crop_rotation",
 "biochar application":"biochar", "charcoal production":"biochar",
 "application of compost":"compost", "addition of topsoil and composts":"compost",
 "composted cotton gin trash":"compost", "livestock manure":"manure",
 "untreated manure":"manure", "ploughing":"tillage", "tillage and harvesting":"tillage",
 # land change
 "deforestation and forest degradation":"deforestation",
 "deforestation and widespread degradation":"deforestation",
 "removal of trees":"deforestation", "fuelwood harvesting":"deforestation",
 "conversion of grassland to cropland":"land_conversion",
 "grassland conversion to cropland":"land_conversion", "land conversion":"land_conversion",
 "reforestation":"afforestation", "vegetation restoration":"restoration",
 "habitat restoration":"restoration", "sand dune stabilisation":"restoration",
 "impervious surfaces":"soil_sealing", "soil sealing":"soil_sealing",
 # biodiversity
 "soil organisms":"soil_biota", "soil microbial community assembly":"soil_biota",
 "earthworm functional diversity":"earthworm_diversity",
 "native wild bees":"pollinators", "pollinator abundance":"pollinators",
 "pollinator diversity":"pollinators", "pollinator species richness":"pollinators",
 "pollinator decline":"pollinators", "wild pollinator survival":"pollinators",
 "invasive_pollinators":"invasive_species", "crop pollination":"pollination_services",
 "species loss":"species_richness", "tree species richness":"species_richness",
 "plant diversity":"plant_diversity", "agro-biodiversity":"species_richness",
 "ecosystem diversity":"habitat_diversity", "pollinator habitat":"pollinator_habitat",
 "pollinator food resources":"pollinator_habitat", "defaunation":"species_richness",
 # chemistry / pollution
 "soil fertility":"soil_fertility", "soil nutrient availability":"nutrient_availability",
 "micronutrient availability":"nutrient_availability", "nutrient cycling":"nutrient_cycling",
 "nitrogen cycling":"nutrient_cycling", "nitrogen retention":"nutrient_cycling",
 "nitrogen loss":"nitrogen_loss", "nitrogen pollution":"pollution_load",
 "heavy metals in urban waste":"pollution_load", "trace metals":"pollution_load",
 "organochlorine pesticides":"pesticide_use", "neonicotinoids":"pesticide_use",
 "herbicide-tolerant crops":"pesticide_use", "pollution":"pollution_load",
 "soluble salts":"salinity", "seawater intrusion":"salinity",
 "n2o flux":"N2O_emissions", "co2 and n2o emissions":"ghg_emissions",
 "greenhouse-gas emissions":"ghg_emissions",
 "rising atmospheric co2":"atmospheric_CO2", "atmospheric co2":"atmospheric_CO2",
 "co2 fertilisation":"atmospheric_CO2", "co2 fertilization":"atmospheric_CO2",
 # cover / structure
 "cover on soil surface":"ground_cover", "ground cover < 50%":"ground_cover",
 "vegetation_cover":"vegetation_cover", "vegetation development":"vegetation_cover",
 "vegetation green-up":"vegetation_cover", "shrub encroachment":"vegetation_cover",
 "soil crusts":"soil_crusts", "soil compaction prevention":"soil_compaction",
 # yield
 "agricultural productivity":"crop_yield", "wheat production":"crop_yield",
 "crop yield stability":"crop_yield", "closing yield gaps":"crop_yield",
 "net primary production":"net_primary_production",
 "human appropriation of net primary production":"net_primary_production",
 "biomass production":"biomass", "livestock feed intake":"biomass",
 # services
 "ecosystem services":"ecosystem_services", "environmental services":"ecosystem_services",
 "wetland ecosystem services":"ecosystem_services",
 "ecosystem multifunctionality":"ecosystem_multifunctionality",
 "pest-control services":"pest_control_services", "resistance to invasion":"resilience",
 "resilience":"resilience", "fire intensity":"fire_frequency",
 "invasive species":"invasive_species", "buffelgrass":"invasive_species",
 "planting of non-native species":"invasive_species",
 "urbanization":"urbanization", "mining":"mining",
}

# ── 3. regex families ────────────────────────────────────────────────
RULES = [
 (r"\b(drip|sprinkler|flood|full|periodic|excess|irrigated)\b.*irrigat|^irrigat", "irrigation"),
 (r"mulch|crop residue|stubble retained|residues on soil surface|maintaining crop residues", "mulching"),
 (r"fertili[sz]", "fertilizer_use"),
 (r"pesticide", "pesticide_use"),
 (r"intercrop|desmodium", "intercropping"),
 (r"deforestat", "deforestation"),
 (r"afforestation|reforestation", "afforestation"),
 (r"agroforest|silvopast|silvi-past|agrisilvi", "agroforestry_agrisilvicultural"),
 (r"fragmentat", "fragmentation"),
 (r"intensif(y|ication)|intensive agricultur", "intensive_agriculture"),
 (r"waterlog|lowering water table|permafrost thaw", "waterlogging"),
]

# nodes deliberately excluded from causal traversal
PERIPHERAL = re.compile(
 r"redd\+|certification|subsid|programme|program\b|scheme|funding|tenure|"
 r"social trust|migration|emigration|employment|human development|conflict|"
 r"damage on local communities|corn prices|livelihood|aquapon|bioenergy|biofuel|"
 r"moose|mongolian|costa rican|gfgp|cyclone|ozone|milder winters|algal bloom|"
 r"food waste|local seed use|watershed projects|green wall|community forestry|"
 r"land-sparing|land management response options|copper|zinc_addition|lime_addition|"
 r"halophyte|napier|enhanced_mineral_weathering|impervious|urban waste", re.I)

def normalise(raw):
    s = re.sub(r"\s+", " ", (raw or "").strip())
    key = s.lower()
    if key in ALIASES:            return ALIASES[key], "alias"
    if s in PROMOTED:             return s, "promoted"
    snake = key.replace(" ", "_").replace("-", "_")
    if snake in PROMOTED:         return snake, "promoted"
    for pat, canon in RULES:
        if re.search(pat, key):   return canon, "rule"
    return s, "unmapped"

# ── 4. apply, dedupe, score ──────────────────────────────────────────
raw_links = []
for f in OUT_DIR.rglob("*.json"):
    if "_v10" in str(f) or "_failures" in str(f): continue
    d = json.loads(f.read_text())
    for l in d.get("links", []):
        l["_unit"] = d["_meta"]["unit_id"]; l["_doc"] = d["_meta"]["doc_id"]
        raw_links.append(l)

edges, how = defaultdict(list), Counter()
for l in raw_links:
    a, ha = normalise(l["from_node"]); b, hb = normalise(l["to_node"])
    how[ha] += 1; how[hb] += 1
    if a == b: continue
    cond = (l.get("condition") or "").strip().lower() or None
    edges[(a, b, l["effect"], cond)].append(l)

graph = []
for (a, b, eff, cond), group in edges.items():
    scope = "peripheral" if (PERIPHERAL.search(a) or PERIPHERAL.search(b)) else "core"
    graph.append({
        "from_node": a, "to_node": b, "effect": eff, "condition": cond,
        "support_count": len(group), "scope": scope,
        "docs": sorted({g["_doc"] for g in group}),
        "units": sorted({g["_unit"] for g in group}),
        "mechanisms": [g["mechanism"] for g in group if g.get("mechanism")][:3],
        "strength": max((g.get("strength_stated","unstated") for g in group),
                        key=lambda s: {"strong":3,"moderate":2,"weak":1,"unstated":0}[s]),
    })

(ROOT / "variable_graph.json").write_text(json.dumps(graph, indent=2))

core = [e for e in graph if e["scope"] == "core"]
supported = [e for e in core if e["support_count"] >= 2]
print(f"raw links {len(raw_links)} -> unique edges {len(graph)}")
print(f"core {len(core)} | peripheral {len(graph)-len(core)} | core with support>=2: {len(supported)}")
print(f"mapping: {dict(how)}")
print(f"distinct nodes: {len({e['from_node'] for e in graph} | {e['to_node'] for e in graph})}")
print(f"conditional core edges: {sum(1 for e in core if e['condition'])}\n")
for e in sorted(core, key=lambda x: -x["support_count"])[:20]:
    c = f"  [{e['condition']}]" if e["condition"] else ""
    print(f"{e['support_count']:>2}x  {e['from_node']:32} -{e['effect'][:3]}-> {e['to_node']:26}{c}")

raw links 646 -> unique edges 519
core 477 | peripheral 42 | core with support>=2: 61
mapping: {'unmapped': 812, 'alias': 211, 'rule': 231, 'promoted': 38}
distinct nodes: 190
conditional core edges: 216

11x  agroforestry_agrisilvicultural   -inc-> SOC                       
 7x  agroforestry_agrisilvicultural   -inc-> nitrogen                  
 6x  intercropping                    -inc-> SOC                       
 6x  intercropping                    -dec-> soil_erosion              
 6x  agroforestry_agrisilvicultural   -dec-> soil_erosion              
 6x  agroforestry_agrisilvicultural   -inc-> nutrient_availability     
 5x  cover_crops                      -dec-> soil_erosion              
 5x  mulching                         -inc-> soil_moisture             
 5x  agroforestry_agrisilvicultural   -dec-> pollution_load            
 5x  agroforestry_agrisilvicultural   -inc-> carbon_sequestration_rate 
 4x  mulching                         -inc-> nutrient_availability     
 4x

In [ ]:
KNOWN = PROMOTED | set(ALIASES.values()) | {
 "SOC","crop_yield","soil_moisture","soil_biota","species_richness","habitat_diversity",
 "nutrient_availability","nitrogen","water_holding_capacity","bulk_density","temperature",
 "rainfall","biomass","carbon_sequestration_rate","pollution_load","pH","erosion",
 "cover_crops","mulching","intercropping","crop_rotation","no_till","reduced_tillage",
 "manure","compost","irrigation","terracing","check_dams","shelterbelts","pollinators",
 "hedges_buffer_strips","grassland_restoration","rotational_grazing","grazing_exclusion",
 "crop_livestock_integration","conservation_agriculture","crop_diversification",
 "agroforestry_agrisilvicultural","agroforestry_silvopastoral","agroforestry_agrosilvopastoral",
 "integrated_nutrient_mgmt","gypsum_amendment","soil_sealing","pollinator_habitat",
 "ecosystem_services","intensive_agriculture","mining","tree_cover"}

leftover = Counter()
for e in graph:
    for n in (e["from_node"], e["to_node"]):
        if n not in KNOWN and e["scope"] == "core":
            leftover[n] += e["support_count"]
print(len(leftover), "truly unmapped core nodes")
for n, c in leftover.most_common(30):
    print(f"{c:>3}  {n}")

69 truly unmapped core nodes
 14  soil_organic_matter
  6  desertification
  6  land degradation
  5  tree canopy
  4  microbial_biomass
  4  conservation agriculture
  4  overgrazing
  3  no-till
  3  improved tillage and residue incorporation
  3  grazing intensity
  3  herbicides
  3  bush_encroachment
  3  crop production
  2  Shisham based plantation
  2  dust storms
  2  surface albedo
  2  land-use change
  2  Parthenium
  2  water harvesting micro catchments
  2  IWM projects
  2  in-field water harvesting
  2  climate change
  2  salinisation
  2  warming
  2  heavy rainfall events
  2  Climate change
  2  Land degradation
  2  wetland restoration
  2  improved_tillage_and_residue
  2  ground cover below 50%


In [ ]:
# ── patches ──────────────────────────────────────────────────────────
EXTRA = {
 "soil_organic_matter":"SOC", "soil organic matter":"SOC",
 "conservation agriculture":"conservation_agriculture",
 "no-till":"no_till", "improved tillage and residue incorporation":"reduced_tillage",
 "improved_tillage_and_residue":"reduced_tillage",
 "overgrazing":"grazing_pressure", "grazing intensity":"grazing_pressure",
 "bush_encroachment":"vegetation_cover", "tree canopy":"tree_cover",
 "microbial_biomass":"soil_biota", "herbicides":"pesticide_use",
 "crop production":"crop_yield", "salinisation":"salinity",
 "land degradation":"land_degradation", "desertification":"desertification",
 "climate change":"climate_change", "warming":"temperature",
 "land-use change":"land_conversion", "ground cover below 50%":"ground_cover",
 "parthenium":"invasive_species", "dust storms":"soil_erosion",
 "heavy rainfall events":"rainfall", "water harvesting micro catchments":"water_harvesting",
 "in-field water harvesting":"water_harvesting", "iwm projects":"water_harvesting",
 "wetland restoration":"restoration",
 "shisham based plantation":"agroforestry_agrisilvicultural",
}
ALIASES.update(EXTRA)
PROMOTED |= {"land_degradation","desertification","climate_change","water_harvesting","tillage"}

# ── normalise (case-insensitive for PROMOTED too) ────────────────────
PROMOTED_LC = {p.lower(): p for p in PROMOTED}

def normalise(raw):
    s = re.sub(r"\s+", " ", (raw or "").strip())
    key = s.lower()
    if key in ALIASES:                 return ALIASES[key], "alias"
    if key in PROMOTED_LC:             return PROMOTED_LC[key], "promoted"
    snake = key.replace(" ", "_").replace("-", "_")
    if snake in PROMOTED_LC:           return PROMOTED_LC[snake], "promoted"
    for pat, canon in RULES:
        if re.search(pat, key):        return canon, "rule"
    return s, "unmapped"

# ── rebuild edges ────────────────────────────────────────────────────
edges = defaultdict(list)
for l in raw_links:
    a, _ = normalise(l["from_node"]); b, _ = normalise(l["to_node"])
    if a == b: continue
    cond = (l.get("condition") or "").strip().lower() or None
    edges[(a, b, l["effect"], cond)].append(l)

graph = []
for (a, b, eff, cond), group in edges.items():
    scope = "peripheral" if (PERIPHERAL.search(a) or PERIPHERAL.search(b)) else "core"
    graph.append({
        "from_node": a, "to_node": b, "effect": eff, "condition": cond,
        "support_count": len(group), "scope": scope,
        "docs": sorted({g["_doc"] for g in group}),
        "units": sorted({g["_unit"] for g in group}),
        "mechanisms": [g["mechanism"] for g in group if g.get("mechanism")][:3],
        "strength": max((g.get("strength_stated","unstated") for g in group),
                        key=lambda s: {"strong":3,"moderate":2,"weak":1,"unstated":0}[s]),
    })

HIGH_TIER = {"ipcc2019_srccl","ipbes2018_ldr","ipbes2016_pollinators",
             "joshi2023_agj","fao2019_sowbfa"}

def keep(e):
    return (e["support_count"] >= 2
            or e["strength"] in {"strong","moderate"}
            or any(d in HIGH_TIER for d in e["docs"]))

traversable = [e for e in graph if e["scope"] == "core" and keep(e)]

(ROOT / "variable_graph.json").write_text(json.dumps(graph, indent=2))
(ROOT / "variable_graph_traversable.json").write_text(json.dumps(traversable, indent=2))

nodes_now = {e["from_node"] for e in graph} | {e["to_node"] for e in graph}
print(f"unique edges {len(graph)} | distinct nodes {len(nodes_now)}")
print(f"traversable {len(traversable)} | conditional {sum(1 for e in traversable if e['condition'])}")
print()
for e in sorted(traversable, key=lambda x: -x["support_count"])[:15]:
    c = f"  [{e['condition'][:30]}]" if e["condition"] else ""
    print(f"{e['support_count']:>2}x {e['strength'][:4]:5} {e['from_node']:30} -{e['effect'][:3]}-> {e['to_node']:24}{c}")

unique edges 505 | distinct nodes 165
traversable 342 | conditional 130

13x stro  agroforestry_agrisilvicultural -inc-> SOC                     
 8x stro  intercropping                  -inc-> SOC                     
 7x stro  agroforestry_agrisilvicultural -inc-> nitrogen                
 7x stro  agroforestry_agrisilvicultural -dec-> soil_erosion            
 6x stro  intercropping                  -dec-> soil_erosion            
 6x stro  agroforestry_agrisilvicultural -inc-> nutrient_availability   
 5x stro  cover_crops                    -dec-> soil_erosion            
 5x stro  mulching                       -inc-> soil_moisture           
 5x stro  agroforestry_agrisilvicultural -dec-> pollution_load          
 5x stro  agroforestry_agrisilvicultural -inc-> carbon_sequestration_rate
 4x unst  mulching                       -inc-> nutrient_availability   
 4x stro  intercropping                  -inc-> soil_moisture           
 4x stro  agroforestry_agrisilvicultural -inc-> cr

In [ ]:
for e in traversable:
    if e["condition"] and any(k in e["condition"] for k in ["arid","dry","water","semi"]):
        print(f"{e['from_node']:30} -{e['effect'][:3]}-> {e['to_node']:22} [{e['condition'][:45]}]")

agroforestry_agrisilvicultural -inc-> crop_yield             [arid conditions]
agroforestry_agrisilvicultural -dec-> SOC                    [dry climates]
intercropping                  -dec-> crop_yield             [competition for resources (e.g. water, light)]
cover_crops                    -dec-> nutrient_availability  [especially in dry years]
soil_crusts                    -dec-> soil_erosion           [drylands]
vegetation_cover               -dec-> soil_erosion           [drylands]
surface albedo                 -dec-> temperature            [drylands]
surface albedo                 -dec-> rainfall               [drylands]
desertification                -dec-> SOC                    [drylands]
desertification                -dec-> crop_yield             [drylands]
rainfall                       -inc-> carbon_sequestration_rate [drylands]
desertification                -dec-> species_richness       [drylands]
grazing_pressure               -inc-> soil_erosion           [dry peri